# CNN Spatio-Temporal Stream — Deepfake Detection

## Two-Stream Late Fusion Architecture (Stream 2 of 2)

This notebook implements a **research-grade Spatio-Temporal CNN** using Xception backbone with **BiLSTM Temporal Aggregation** for deepfake detection.

### Key Research Contributions:

1. **Temporal Modeling (BiLSTM + Multi-Head Attention)**
   - Unlike naive frame averaging, we model inter-frame dependencies
   - Detects temporal flickering, blending shifts, and motion anomalies
   - Enables detection of GAN/Diffusion artifacts that manifest across frames

2. **Grad-CAM Visualization**
   - Provides visual proof of what the model learns
   - Shows attention on facial regions (jawline, eyes, blending boundaries)
   - Essential for research paper methodology section

```
┌──────────────────────────────────────────────────────────────────────────────┐
│                    SPATIO-TEMPORAL ARCHITECTURE                              │
├──────────────────────────────────────────────────────────────────────────────┤
│                                                                              │
│   Video (T frames) ──→ MTCNN Face Detection ──→ T × (224, 224, 3) crops     │
│                                                                              │
│   ┌─────────────────────────────────────────────────────────────────────┐   │
│   │  SPATIAL FEATURE EXTRACTION                                          │   │
│   │  EfficientNet-B4 (pretrained, 1792-dim features per frame)          │   │
│   │                                                                      │   │
│   │      Frame 1 ──→ [f₁]                                               │   │
│   │      Frame 2 ──→ [f₂]     ──→ Feature Sequence (T × 1792)           │   │
│   │      ...                                                             │   │
│   │      Frame T ──→ [fₜ]                                               │   │
│   └─────────────────────────────────────────────────────────────────────┘   │
│                              ↓                                               │
│   ┌─────────────────────────────────────────────────────────────────────┐   │
│   │  TEMPORAL AGGREGATION (BiLSTM + Multi-Head Attention)               │   │
│   │                                                                      │   │
│   │  [f₁, f₂, ..., fₜ] ──→ BiLSTM (2-layer, bidirectional)              │   │
│   │                                ↓                                     │   │
│   │                    Multi-Head Self-Attention (4 heads)               │   │
│   │                                ↓                                     │   │
│   │                    Weighted Temporal Pooling                         │   │
│   │                                ↓                                     │   │
│   │                    Video-Level Representation (512-dim)              │   │
│   └─────────────────────────────────────────────────────────────────────┘   │
│                              ↓                                               │
│   ┌─────────────────────────────────────────────────────────────────────┐   │
│   │  CLASSIFIER                                                          │   │
│   │  Linear(512 → 256) → BatchNorm → GELU → Dropout                     │   │
│   │  Linear(256 → 128) → BatchNorm → GELU → Dropout                     │   │
│   │  Linear(128 → 1) → Sigmoid                                          │   │
│   │                        ↓                                             │   │
│   │                    P_CNN (0 = Real, 1 = Fake)                        │   │
│   └─────────────────────────────────────────────────────────────────────┘   │
│                                                                              │
└──────────────────────────────────────────────────────────────────────────────┘

                    ↓ LATE FUSION ↓

        P_final = w₁ × P_CNN + w₂ × P_rPPG (from Stream 1)
```

**Output:** `cnn_predictions.csv` with video-level P_CNN scores for Late Fusion

In [ ]:
import os

root = "/kaggle/input"

for current_path, dirs, files in os.walk(root):
    level = current_path.replace(root, "").count(os.sep)
    indent = "    " * level
    print(f"{indent}{os.path.basename(current_path)}/")

## 1. Setup & Imports

In [ ]:
!pip install "numpy<2" --force-reinstall

In [ ]:
# timm install handled in Cell 5
pass

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# INSTALL DEPENDENCIES
# ═══════════════════════════════════════════════════════════════════════════════

import subprocess
import sys

def install_packages():
    packages = [
        "facenet-pytorch",
        "timm==0.9.16",
        "albumentations>=1.4.0,<2.0.0",
        "seaborn>=0.12.0",
        "opencv-python-headless",
        "scikit-learn>=1.2.0",  # ISSUE 4 FIX: StratifiedGroupKFold shuffle parameter
    ]
    for pkg in packages:
        print(f"Installing {pkg}...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)
    print("✓ All packages installed")

install_packages()

DATA LOADING

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# UNIFIED DATA COMPILER — Run ONCE before all other cells
# Creates master_dataset_index.csv used by both CNN + rPPG notebooks
# Guarantees: alignment, balance, P100-safe RAM, no metadata bottleneck
# ═══════════════════════════════════════════════════════════════════════

import os, json, random
import pandas as pd
random.seed(42)

OUTPUT_CSV    = "/kaggle/working/master_dataset_index.csv"
VIDEO_EXTS    = ('.mp4', '.avi', '.mov', '.mkv', '.webm')
MAX_PER_CLASS = 200  # per source — keeps total ~1600 videos, safe for P100

def scan_folder(folder, label, source, max_n=MAX_PER_CLASS):
    if not os.path.exists(folder):
        print(f"  ⚠️  NOT FOUND: {folder}")
        return []
    # FIXED: os.walk() is recursive — finds videos in subdirectories
    files = []
    for root, dirs, fnames in os.walk(folder):
        for f in fnames:
            if f.lower().endswith(VIDEO_EXTS):
                files.append(os.path.join(root, f))
    files = sorted(files)
    if len(files) > max_n:
        files = random.sample(files, max_n)
    # FIXED: use relative path in video_id to prevent basename collisions
    records = [{'video_id': f"{source}__{os.path.relpath(f, folder).replace(os.sep, '_')}",
                'path':     f,
                'label':    label,
                'source':   source} for f in files]
    print(f"  ✓ {source:30s} | label={label} | {len(records):4d} videos")
    return records

# ── MUST initialize all_records BEFORE any scan calls ────────────────
all_records = []

# ── 1. FaceForensics++ ───────────────────────────────────────────────
print("\n── FaceForensics++ ─────────────────────────────────────────────")
FF_BASE = "/kaggle/input/datasets/xdxd003/ff-c23/FaceForensics++_C23"
all_records += scan_folder(f"{FF_BASE}/original",          0, "FF_real")
all_records += scan_folder(f"{FF_BASE}/Deepfakes",         1, "FF_Deepfakes")
all_records += scan_folder(f"{FF_BASE}/Face2Face",         1, "FF_Face2Face")
all_records += scan_folder(f"{FF_BASE}/FaceSwap",          1, "FF_FaceSwap")
all_records += scan_folder(f"{FF_BASE}/NeuralTextures",    1, "FF_NeuralTextures")
all_records += scan_folder(f"{FF_BASE}/FaceShifter",       1, "FF_FaceShifter")
# FIXED: DeepFakeDetection now placed AFTER initialization, in correct position
all_records += scan_folder(f"{FF_BASE}/DeepFakeDetection", 1, "FF_DeepFakeDetection")

# ── 2. Celeb-DF v2 ───────────────────────────────────────────────────
print("\n── Celeb-DF v2 ─────────────────────────────────────────────────")
CELEB_BASE = "/kaggle/input/datasets/reubensuju/celeb-df-v2"
all_records += scan_folder(f"{CELEB_BASE}/Celeb-real",      0, "Celeb_real",  max_n=150)
all_records += scan_folder(f"{CELEB_BASE}/YouTube-real",    0, "YT_real",     max_n=50)
all_records += scan_folder(f"{CELEB_BASE}/Celeb-synthesis", 1, "Celeb_fake",  max_n=200)

# ── 3. Custom Dataset (lalith023) ────────────────────────────────────
print("\n── Custom Dataset (400 Videos) ─────────────────────────────────")
CUSTOM_BASE = "/kaggle/input/datasets/lalith023/400videos/content/drive/MyDrive/face_dataset_dip"
all_records += scan_folder(f"{CUSTOM_BASE}/real_videos",     0, "Custom_real", max_n=400)
all_records += scan_folder(f"{CUSTOM_BASE}/deepfake_videos", 1, "Custom_fake", max_n=400)

# ── 4. DFDC ──────────────────────────────────────────────────────────
print("\n── DFDC ────────────────────────────────────────────────────────")
DFDC_DIR = "/kaggle/input/datasets/lalith023/dfdc-sample-videos"
dfdc_real, dfdc_fake = [], []
if os.path.exists(DFDC_DIR):
    meta_path = os.path.join(DFDC_DIR, "metadata.json")
    if os.path.exists(meta_path):
        with open(meta_path, 'r') as f:
            meta = json.load(f)
        for filename, info in meta.items():
            path = os.path.join(DFDC_DIR, filename)
            if not os.path.exists(path):
                continue
            rec = {'video_id': f"DFDC__{filename}",
                   'path':     path,
                   'label':    1 if info['label'] == 'FAKE' else 0,
                   'source':   'DFDC'}
            if info['label'] == 'REAL':
                dfdc_real.append(rec)
            else:
                dfdc_fake.append(rec)
        n = len(dfdc_real)
        all_records += dfdc_real
        if len(dfdc_fake) > 0 and n > 0:
            all_records += random.sample(dfdc_fake, min(n, len(dfdc_fake)))
        if n == 0:
            print(f"  ⚠️  DFDC has 0 real videos — skipping DFDC fake videos")
        print(f"  ✓ {'DFDC':30s} | real={len(dfdc_real)} | fake={min(n, len(dfdc_fake))}")
    else:
        print(f"  ⚠️  metadata.json not found in {DFDC_DIR}")
else:
    print(f"  ⚠️  DFDC dir not found: {DFDC_DIR}")

# Guard: fail loudly if data is critically missing
assert len(all_records) >= 500, \
    f"FATAL: Only {len(all_records)} total videos found — need at least 500. Check all dataset paths above!"
n_real_pre = sum(1 for r in all_records if r['label'] == 0)
n_fake_pre = sum(1 for r in all_records if r['label'] == 1)
assert n_real_pre >= 50, f"FATAL: Only {n_real_pre} real videos — check real dataset paths!"
assert n_fake_pre >= 50, f"FATAL: Only {n_fake_pre} fake videos — check fake dataset paths!"

# ── Calculate class weights instead of discarding samples ────────────
df     = pd.DataFrame(all_records)
n_real = len(df[df['label'] == 0])
n_fake = len(df[df['label'] == 1])
print(f"\n── Dataset Balance ─────────────────────────────────────────────")
print(f"  Real : {n_real}")
print(f"  Fake : {n_fake}")

# Calculate class imbalance ratio for reference
# Note: FocalLoss with alpha=0.5 inherently handles class weighting
# by applying balanced weights to both classes
n_ratio = n_real / n_fake if n_fake > 0 else 1.0
print(f"  Real/Fake ratio: {n_ratio:.2f}")
print(f"  ✓ FocalLoss alpha is handled via Config")
print(f"  ✓ All samples retained (no discarding)")

# Shuffle the dataframe
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

# ── Verify every path actually exists ────────────────────────────────
missing = df[~df['path'].apply(os.path.exists)]
if len(missing) > 0:
    print(f"\n  ⚠️  Removing {len(missing)} missing paths")
    df = df[df['path'].apply(os.path.exists)].reset_index(drop=True)

# ── Save ─────────────────────────────────────────────────────────────
df.to_csv(OUTPUT_CSV, index=False)

print(f"\n{'='*60}")
print(f"✅ master_dataset_index.csv saved → {OUTPUT_CSV}")
print(f"{'='*60}")
print(f"  Total  : {len(df)}")
print(f"  Real   : {len(df[df['label']==0])}")
print(f"  Fake   : {len(df[df['label']==1])}")
print(f"  Missing: {len(missing)}")
print(f"\n  Source breakdown:")
print(df.groupby(['source','label']).size().to_string())
print(f"\n  ✓ Both CNN and rPPG notebooks read from this file.")
print(f"  ✓ video_id is globally unique — fusion alignment guaranteed.")
print(f"{'='*60}")


── FaceForensics++ ─────────────────────────────────────────────
  ✓ FF_real                        | label=0 |  200 videos
  ✓ FF_Deepfakes                   | label=1 |  200 videos
  ✓ FF_Face2Face                   | label=1 |  200 videos
  ✓ FF_FaceSwap                    | label=1 |  200 videos
  ✓ FF_NeuralTextures              | label=1 |  200 videos
  ✓ FF_FaceShifter                 | label=1 |  200 videos
  ✓ FF_DeepFakeDetection           | label=1 |  200 videos

── Celeb-DF v2 ─────────────────────────────────────────────────
  ✓ Celeb_real                     | label=0 |  150 videos
  ✓ YT_real                        | label=0 |   50 videos
  ✓ Celeb_fake                     | label=1 |  200 videos

── Custom Dataset (400 Videos) ─────────────────────────────────
  ✓ Custom_real                    | label=0 |  400 videos
  ✓ Custom_fake                    | label=1 |  400 videos

── DFDC ────────────────────────────────────────────────────────
  ✓ DFDC                    

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# IMPORTS (P100 COMPATIBLE - STRICT FP32, NO AMP)
# ═══════════════════════════════════════════════════════════════════════════════

import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'max_split_size_mb:128'

import re
import gc
import cv2
import json
import math
import random
import warnings
import numpy as np
import pandas as pd
from collections import defaultdict
from typing import List, Tuple, Dict, Optional
from tqdm.auto import tqdm
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim.swa_utils import AveragedModel, update_bn, SWALR

import timm
from facenet_pytorch import MTCNN
import albumentations as A

from sklearn.model_selection import (train_test_split, StratifiedKFold,
                                     GroupKFold, StratifiedGroupKFold)
from sklearn.metrics import (accuracy_score, roc_auc_score, f1_score,
                             classification_report, roc_curve,
                             precision_score, recall_score,
                             average_precision_score)

import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
# Issue 16: Remove deterministic mode to avoid 9-hour timeout
# torch.backends.cudnn.deterministic = True  # REMOVED - causes ~40% slowdown
torch.backends.cudnn.benchmark = True  # Enable for faster training

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
    cap = torch.cuda.get_device_capability(0)
    print(f"Compute Capability: {cap[0]}.{cap[1]}")
    if cap[0] < 7:
        print("⚠️ PASCAL GPU — Strict FP32")
print("✓ All imports successful")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CONFIGURATION — P100 Optimized + ALL ENHANCEMENTS
# ═══════════════════════════════════════════════════════════════════════════════

import math, re

class Config:
    EXPERIMENT_NAME    = "CNN_Xception_BiLSTM_Attn_AllEnhancements"
    EXPERIMENT_VERSION = "v6.0_maximum_accuracy"

    # ── Dataset ──────────────────────────────────────────────────────
    MASTER_CSV  = "/kaggle/working/master_dataset_index.csv"
    OUTPUT_DIR  = "/kaggle/working"
    FACE_CACHE  = "/kaggle/working/face_cache"

    # ── Frame extraction ─────────────────────────────────────────────
    FRAMES_PER_VIDEO = 15     # safe with BATCH=2, Xception, P100
    IMG_SIZE         = 299    # Xception requirement

    # ── P100 memory ──────────────────────────────────────────────────
    BATCH_SIZE              = 2   # Safe limit to prevent OOM
    GRAD_ACCUMULATION_STEPS = 4   # Effective batch size = 8 for stable SAM gradients
    NUM_WORKERS             = 0

    # ── Training ─────────────────────────────────────────────────────
    NUM_EPOCHS    = 40        # more epochs with warm restarts
    LEARNING_RATE = 1e-4      # lower LR — more stable with Xception
    WEIGHT_DECAY  = 1e-4
    WARMUP_RATIO  = 0.1      # shorter warmup — real learning starts epoch 2

    # ── Loss ─────────────────────────────────────────────────────────
    FOCAL_ALPHA     = 0.5     # IMPROVEMENT: 0.5 is balanced for 50/50 dataset
    FOCAL_GAMMA     = 2.0
    LABEL_SMOOTHING = 0.1     # IMPROVEMENT: 0.1 for better calibration with noisy labels

    # ── Model ────────────────────────────────────────────────────────
    MODEL_NAME      = "xception"
    DROPOUT         = 0.3
    HIDDEN_DIM      = 256
    TEMPORAL_TYPE   = "bilstm_attention"
    LSTM_HIDDEN     = 256
    LSTM_LAYERS     = 2
    ATTENTION_HEADS = 4
    FREEZE_BACKBONE = False  # Note: Backbone is always frozen initially, unfrozen at UNFREEZE_EPOCH

    # ── Curriculum / Progressive training ───────────────────────────
    UNFREEZE_EPOCH        = 5    # unfreeze backbone at epoch 5
    HARD_MINING_EPOCH     = 10   # start hard negative mining at epoch 10
    MIXUP_ALPHA           = 0.2  # mixup blend strength
    USE_MIXUP             = True
    USE_PROGRESSIVE_FRAMES = True  # start with 5 frames, grow to 15

    # ── SWA ──────────────────────────────────────────────────────────
    USE_SWA       = True
    SWA_START     = 15            # FIX: Earlier start to ensure SWA gets 5+ epochs
    SWA_LR        = 5e-5         # SWA learning rate

    # ── Splits ───────────────────────────────────────────────────────
    # ISSUE 4 NOTE: This is a SINGLE-FOLD experiment (fold 0 only).
    # K_FOLDS=5 defines the split strategy but no fold loop exists.
    # For full 5-fold CV, run this notebook 5 times with CURRENT_FOLD=0,1,2,3,4
    # and report mean±std across folds in the paper.
    K_FOLDS            = 5
    CURRENT_FOLD       = 0   # Change to 1,2,3,4 for other folds
    USE_IDENTITY_SPLIT = True
    TRAIN_RATIO        = 0.8
    VAL_RATIO          = 0.2

    # ── Early Stopping ───────────────────────────────────────────────
    PATIENCE = 25  # FIX: More patience to ensure SWA gets decent snapshots

    @classmethod
    def to_dict(cls):
        return {k: v for k, v in vars(cls).items()
                if not k.startswith('_') and not callable(v)}

cfg = Config()
os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)
os.makedirs(cfg.FACE_CACHE, exist_ok=True)

import json
with open(os.path.join(cfg.OUTPUT_DIR, 'config.json'), 'w') as f:
    json.dump(cfg.to_dict(), f, indent=2, default=str)

print("="*70)
print(f"  Backbone        : {cfg.MODEL_NAME}")
print(f"  Frames/vid      : {cfg.FRAMES_PER_VIDEO} @ {cfg.IMG_SIZE}px")
print(f"  Effective batch : {cfg.BATCH_SIZE*cfg.GRAD_ACCUMULATION_STEPS}")
print(f"  LR              : {cfg.LEARNING_RATE}")
print(f"  Label smoothing : {cfg.LABEL_SMOOTHING}")
print(f"  Unfreeze epoch  : {cfg.UNFREEZE_EPOCH}")
print(f"  Hard mining     : epoch {cfg.HARD_MINING_EPOCH}+")
print(f"  SWA             : epoch {cfg.SWA_START}+")
print(f"  Epochs          : {cfg.NUM_EPOCHS}")
print("="*70)

## 2. Frame Extraction & Face Detection

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# MTCNN FACE DETECTOR WITH ALIGNMENT + QUALITY FILTERING
# ═══════════════════════════════════════════════════════════════════════════════

class FaceExtractor:
    def __init__(self, device, img_size=299, margin=40):
        self.device   = device
        self.img_size = img_size
        self.margin   = margin

        self.mtcnn = MTCNN(
            image_size=img_size, margin=margin,
            min_face_size=60, thresholds=[0.6, 0.7, 0.7],
            factor=0.709, post_process=False,
            device=device, keep_all=False,
        )
        print(f"✓ MTCNN initialized on {device} (size={img_size})")

    def extract_face_aligned(self, frame: np.ndarray) -> Optional[np.ndarray]:
        """
        Extract face WITH eye-landmark alignment.
        Aligned faces eliminate pose variation — model focuses purely
        on manipulation artifacts rather than head orientation.
        """
        try:
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            pil_img   = Image.fromarray(frame_rgb)

            boxes, probs, landmarks = self.mtcnn.detect(
                pil_img, landmarks=True)

            if (landmarks is None or len(landmarks) == 0
                    or probs[0] is None or probs[0] < 0.9):
                # Low confidence — fall back to center crop
                return self._center_crop(frame_rgb)

            lm        = landmarks[0]
            left_eye  = lm[0]
            right_eye = lm[1]

            # Calculate alignment angle
            dy    = right_eye[1] - left_eye[1]
            dx    = right_eye[0] - left_eye[0]
            angle = np.degrees(np.arctan2(dy, dx))

            # Only align if tilt is significant (>2 degrees)
            if abs(angle) > 2.0:
                eye_center = (
                    float((left_eye[0] + right_eye[0]) / 2),
                    float((left_eye[1] + right_eye[1]) / 2)
                )
                h, w = frame_rgb.shape[:2]
                M = cv2.getRotationMatrix2D(eye_center, angle, 1.0)
                frame_rgb = cv2.warpAffine(
                    frame_rgb, M, (w, h), flags=cv2.INTER_LINEAR)
                pil_img = Image.fromarray(frame_rgb)

            face = self.mtcnn(pil_img)
            if face is not None:
                face_np = face.permute(1,2,0).cpu().numpy().astype(np.uint8)
                # Quality check — reject blurry faces
                gray    = cv2.cvtColor(face_np, cv2.COLOR_RGB2GRAY)
                blur    = cv2.Laplacian(gray, cv2.CV_64F).var()
                if blur < 20.0:   # too blurry — use center crop instead
                    return self._center_crop(frame_rgb)
                return face_np
            else:
                return self._center_crop(frame_rgb)

        except Exception:
            try:
                return self._center_crop(
                    cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            except Exception:
                return None

    def extract_face(self, frame: np.ndarray) -> Optional[np.ndarray]:
        return self.extract_face_aligned(frame)

    def _center_crop(self, frame: np.ndarray) -> np.ndarray:
        h, w   = frame.shape[:2]
        size   = min(h, w)
        y      = (h - size) // 2
        x      = (w - size) // 2
        crop   = frame[y:y+size, x:x+size]
        return cv2.resize(crop, (self.img_size, self.img_size))

face_extractor = FaceExtractor(DEVICE, img_size=cfg.IMG_SIZE)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# VIDEO FRAME EXTRACTION
# ═══════════════════════════════════════════════════════════════════════════════

def extract_frames_from_video(video_path: str, n_frames: int = 10) -> List[np.ndarray]:
    """
    Extract n evenly spaced frames from a video.
    
    Args:
        video_path: Path to video file
        n_frames: Number of frames to extract
        
    Returns:
        List of frame arrays (BGR format)
    """
    cap = cv2.VideoCapture(video_path)
    
    if not cap.isOpened():
        return []
    
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total_frames <= 0:
        cap.release()
        return []  # Corrupted video
    
    if total_frames < n_frames:
        # If video has fewer frames, extract all
        frame_indices = list(range(total_frames))
    else:
        # Evenly spaced frame indices
        frame_indices = np.linspace(0, total_frames - 1, n_frames, dtype=int)
    
    frames = []
    for idx in frame_indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if ret:
            frames.append(frame)
    
    cap.release()
    return frames


def process_video(video_path: str, face_extractor: FaceExtractor, n_frames: int = 10) -> List[np.ndarray]:
    """
    Extract frames and detect faces from a video.
    BUG 5 FIX: depends on FaceExtractor.extract_face which is defined in Cell 12. Ensure Cell 12 is run first.
    
    Returns:
        List of face crops as numpy arrays
    """
    frames = extract_frames_from_video(video_path, n_frames)
    
    face_crops = []
    for frame in frames:
        face = face_extractor.extract_face(frame)
        if face is not None:
            face_crops.append(face)
    
    return face_crops


print("✓ Frame extraction functions defined")

✓ Frame extraction functions defined


In [23]:
# ═══════════════════════════════════════════════════════════════════════
# LOAD VIDEOS FROM MASTER CSV (replaces collect_video_paths)
# ═══════════════════════════════════════════════════════════════════════

def load_videos_from_csv(csv_path: str) -> List[dict]:
    """
    Load pre-compiled, pre-balanced, path-verified video list.
    Both CNN and rPPG notebooks read from this identical file.
    Alignment for Late Fusion is mathematically guaranteed.
    """
    if not os.path.exists(csv_path):
        raise FileNotFoundError(
            f"master_dataset_index.csv not found at: {csv_path}\n"
            f"You MUST run the Compiler cell first before this cell."
        )

    df = pd.read_csv(csv_path)

    # Verify required columns
    required = ['video_id', 'path', 'label', 'source']
    for col in required:
        if col not in df.columns:
            raise ValueError(f"Missing column '{col}' in CSV.")

    # Verify paths still exist
    missing = df[~df['path'].apply(os.path.exists)]
    if len(missing) > 0:
        print(f"  ⚠️  {len(missing)} paths missing — skipping them")
        df = df[df['path'].apply(os.path.exists)].reset_index(drop=True)

    videos = df.to_dict('records')

    print("\n" + "="*70)
    print("DATASET LOADED FROM MASTER CSV")
    print("="*70)
    print(f"  Total  : {len(videos)}")
    print(f"  Real   : {sum(1 for v in videos if v['label']==0)}")
    print(f"  Fake   : {sum(1 for v in videos if v['label']==1)}")
    print(f"\n  Source breakdown:")
    print(df.groupby(['source','label']).size().to_string())
    print("="*70)
    return videos

all_videos = load_videos_from_csv(cfg.MASTER_CSV)
print(f"\nTotal videos: {len(all_videos)}")
print(f"  Real: {sum(1 for v in all_videos if v['label']==0)}")
print(f"  Fake: {sum(1 for v in all_videos if v['label']==1)}")


DATASET LOADED FROM MASTER CSV
  Total  : 1754
  Real   : 877
  Fake   : 877

  Source breakdown:
source                label
Celeb_fake            1         97
Celeb_real            0        150
Custom_fake           1        194
Custom_real           0        400
DFDC                  0         77
                      1         36
FF_DeepFakeDetection  1         81
FF_Deepfakes          1         87
FF_Face2Face          1        100
FF_FaceShifter        1         91
FF_FaceSwap           1        101
FF_NeuralTextures     1         90
FF_real               0        200
YT_real               0         50

Total videos: 1754
  Real: 877
  Fake: 877


In [24]:
# ═══════════════════════════════════════════════════════════════════════
# DISK-BASED FACE EXTRACTION — P100 RAM-SAFE
# Saves each video's faces as .npy file immediately after extraction.
# Never holds more than 1 video worth of faces in RAM at once.
# 1600 videos × 20 frames would need ~8GB RAM — this avoids that.
# ═══════════════════════════════════════════════════════════════════════

def extract_and_cache_faces(videos: List[dict],
                             face_extractor: FaceExtractor,
                             cache_dir: str,
                             n_frames: int = 20) -> dict:
    """
    Extract faces video-by-video and save to disk immediately.
    Returns: cache_index dict {video_id: cache_path}
    RAM stays flat — only 1 video in memory at any time.
    """
    os.makedirs(cache_dir, exist_ok=True)
    cache_index = {}
    failed      = []

    print("\n" + "="*70)
    print("EXTRACTING & CACHING FACES (DISK-BASED)")
    print("="*70)
    print(f"  Cache dir : {cache_dir}")
    print(f"  Videos    : {len(videos)}")
    print(f"  Frames/vid: {n_frames}")
    print("  (Skips already-cached videos automatically)")
    print("="*70)

    for video in tqdm(videos, desc="Extracting faces"):
        video_id   = video['video_id']
        video_path = video['path']

        # Safe filename (video_id may contain __ separators)
        safe_name  = video_id.replace("/", "_").replace("\\", "_")
        cache_path = os.path.join(cache_dir, f"{safe_name}.npy")

        # Skip if already cached from a previous run
        if os.path.exists(cache_path):
            cache_index[video_id] = cache_path
            continue

        # Extract faces for this video only
        faces = process_video(video_path, face_extractor, n_frames)

        if len(faces) >= 3:
            faces_array = np.array(faces, dtype=np.uint8)
            np.save(cache_path, faces_array)
            cache_index[video_id] = cache_path
        else:
            failed.append(video_id)

        # Free this video's data immediately
        del faces
        if len(cache_index) % 100 == 0:
            gc.collect()

    print(f"\n✓ Cached : {len(cache_index)} videos")
    print(f"✗ Failed : {len(failed)} videos")

    if len(failed) > 0:
        print(f"\n  Failed videos (first 10): {failed[:10]}")

    # Final cleanup
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return cache_index

# Run extraction — faces saved to disk, not RAM
cache_index = extract_and_cache_faces(
    all_videos,
    face_extractor,
    cache_dir=cfg.FACE_CACHE,
    n_frames=cfg.FRAMES_PER_VIDEO
)

# Save cache index for recovery
cache_index_path = os.path.join(cfg.OUTPUT_DIR, "cache_index.json")
with open(cache_index_path, 'w') as f:
    json.dump(cache_index, f)
print(f"\n✓ Cache index saved: {cache_index_path}")
print(f"  Videos with valid faces: {len(cache_index)}/{len(all_videos)}")


EXTRACTING & CACHING FACES (DISK-BASED)
  Cache dir : /kaggle/working/face_cache
  Videos    : 1754
  Frames/vid: 10
  (Skips already-cached videos automatically)


Extracting faces:   0%|          | 0/1754 [00:00<?, ?it/s]


✓ Cached : 1753 videos
✗ Failed : 1 videos

  Failed videos (first 10): ['Celeb_real__id27_0005.mp4']

✓ Cache index saved: /kaggle/working/cache_index.json
  Videos with valid faces: 1753/1754


In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# FACE CACHE VERIFICATION
# (Replaces old "save faceS" cell — faces already saved during extraction)
# ═══════════════════════════════════════════════════════════════════════

import os
import numpy as np

# Verify cache is healthy
npy_files = [f for f in os.listdir(cfg.FACE_CACHE) if f.endswith('.npy')]
total_size_gb = sum(
    os.path.getsize(os.path.join(cfg.FACE_CACHE, f))
    for f in npy_files
) / 1e9

print("="*70)
print("FACE CACHE STATUS")
print("="*70)
print(f"  Cached files : {len(npy_files)}")
print(f"  Total size   : {total_size_gb:.2f} GB")
print(f"  Cache dir    : {cfg.FACE_CACHE}")

# Quick sanity check on one file
if len(npy_files) > 0:
    sample_path = os.path.join(cfg.FACE_CACHE, npy_files[0])
    sample      = np.load(sample_path)
    print(f"\n  Sample shape : {sample.shape}  "
          f"(frames × H × W × channels)")
    print(f"  dtype        : {sample.dtype}")
    print(f"  Value range  : [{sample.min()}, {sample.max()}]")
    
    # ISSUE 9 FIX: Validate cached face resolution matches cfg.IMG_SIZE
    if sample.shape[1] != cfg.IMG_SIZE or sample.shape[2] != cfg.IMG_SIZE:
        raise ValueError(
            f"STALE CACHE DETECTED: Cached faces are {sample.shape[1]}x{sample.shape[2]} "
            f"but cfg.IMG_SIZE={cfg.IMG_SIZE}. "
            f"Delete {cfg.FACE_CACHE}/ and re-run face extraction!"
        )
    if sample.shape[0] < cfg.FRAMES_PER_VIDEO:
        raise ValueError(
            f"STALE CACHE DETECTED: Cached {sample.shape[0]} frames but cfg.FRAMES_PER_VIDEO={cfg.FRAMES_PER_VIDEO}. "
            f"Delete {cfg.FACE_CACHE}/ and re-extract!"
        )
    print(f"  Resolution   : {sample.shape[1]}x{sample.shape[2]} ✓ (matches cfg.IMG_SIZE)")
    del sample

print("="*70)
print("✓ Face cache verified — ready for training")
print("✓ RAM usage: minimal (faces are on disk, not in memory)")
print("="*70)

FACE CACHE STATUS
  Cached files : 1753
  Total size   : 5.28 GB
  Cache dir    : /kaggle/working/face_cache

  Sample shape : (20, 224, 224, 3)  (frames × H × W × channels)
  dtype        : uint8
  Value range  : [0, 255]
✓ Face cache verified — ready for training
✓ RAM usage: minimal (faces are on disk, not in memory)


In [26]:
# ═══════════════════════════════════════════════════════════════════════
# FREE GPU MEMORY — Delete MTCNN before loading training model
# CRITICAL: MTCNN holds ~15GB GPU memory after extraction
# Must be freed before Xception can be loaded
# ═══════════════════════════════════════════════════════════════════════

import gc, torch

# Delete MTCNN face extractor from GPU
if 'face_extractor' in dir():
    del face_extractor

# Force GPU memory release
gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()

# Confirm how much is now free
free_gb = (torch.cuda.get_device_properties(0).total_memory - 
           torch.cuda.memory_allocated(0)) / 1e9
print(f"✓ MTCNN deleted and GPU memory freed")
print(f"✓ GPU memory now free: {free_gb:.1f} GB")
print(f"✓ Safe to load Xception model")

✓ MTCNN deleted and GPU memory freed
✓ GPU memory now free: 16.6 GB
✓ Safe to load EfficientNet-B4 model


## 3. Dataset Creation & Data Leakage Prevention

In [27]:
# ═══════════════════════════════════════════════════════════════════════════════
# IDENTITY-BASED DATA SPLITTING (CRITICAL: Prevents Data Leakage)
# ═══════════════════════════════════════════════════════════════════════════════
#
# PROBLEM: Random train/test split causes IDENTITY LEAKAGE:
#   - Real Person A in training, Fake Person A in validation
#   - Model memorizes faces, not deepfake artifacts
#   - Inflated metrics that don't generalize
#
# SOLUTION: Group videos by source identity BEFORE splitting:
#   - All videos of Person A go to train OR val, never both
#   - Forces model to learn manipulation artifacts, not faces
#   - Required by IEEE T-IFS, CVPR, and top security venues
#
# ═══════════════════════════════════════════════════════════════════════════════

# sklearn imports already done in cell 8
from collections import defaultdict


def extract_identity_from_filename(filename: str) -> str:
    """
    Extract identity/person ID from video filename.
    
    Common deepfake dataset naming conventions:
    - FaceForensics++: {id}_{seq}.mp4 → identity = id
    - Celeb-DF: id{num}_{seq}.mp4 → identity = id{num}
    - DFDC: {subject_id}_{video_id}.mp4 → identity = subject_id
    - Custom: video_{person}_{seq}.mp4 → identity = person
    
    Fallback: Use first segment before underscore/dash
    
    Args:
        filename: Video filename (with or without extension)
    
    Returns:
        String identifier for the source identity
    """
    # Remove extension
    name = os.path.splitext(os.path.basename(filename))[0]
    
    # Common patterns for identity extraction
    patterns = [
        # FaceForensics++: 123_456.mp4 → 123
        r'^(\d+)_\d+$',
        # Celeb-DF: id0_0000.mp4 → id0
        r'^(id\d+)_\d+$',  # Celeb-DF: id0_0000.mp4 → id0
        # DFDC style: abcdef_0.mp4 → abcdef
        r'^([a-zA-Z]+\d*)_\d+$',
        # General: person1_video2.mp4 → person1
        r'^([^_]+)_.*$',
    ]
    
    for pattern in patterns:
        match = re.match(pattern, name)
        if match:
            return match.group(1)
    
    # Fallback: first part before underscore or full name
    parts = re.split(r'[_\-]', name)
    return parts[0] if parts else name


def extract_identity_from_path(video_path: str, label: int) -> str:
    """
    Extract identity from video path, considering label.
    
    For deepfake datasets, the identity is typically embedded in the path:
    - /real_videos/person1/video.mp4 → person1_real
    - /fake_videos/person1_to_person2/video.mp4 → person1_fake
    
    FIXED Issue 12: FF++ videos are stored in numeric folders (000, 001, etc.)
    which are sequence folders, NOT person identities. The actual identity
    comes from the filename (e.g., 123_456.mp4 -> identity 123).
    """
    filename = os.path.basename(video_path)
    identity = extract_identity_from_filename(filename)
    
    # Get parent folder
    parent = os.path.basename(os.path.dirname(video_path))
    
    # List of known non-identity folders
    exclude_folders = [
        'real_videos', 'fake_videos', 'deepfake_videos',
        'original', 'Deepfakes', 'Face2Face', 'FaceSwap',
        'NeuralTextures', 'FaceShifter', 'DeepFakeDetection',
        'Celeb-real', 'Celeb-synthesis', 'YouTube-real',
        'train_sample_videos', 'videos', 'c23', 'c40'
    ]
    
    # Issue 12 FIX: Also exclude numeric folders (FF++ uses 000, 001, etc.)
    # These are sequence folders, not person identities
    is_numeric_folder = parent.isdigit() if parent else False
    
    if parent and parent not in exclude_folders and not is_numeric_folder:
        identity = f"{parent}_{identity}"
    
    return identity


def assign_identities_to_videos(videos: List[dict]) -> List[dict]:
    """
    Assign identity labels to each video for group-based splitting.
    
    Args:
        videos: List of video dicts with 'video_id', 'label', 'path'
    
    Returns:
        Same list with 'identity' field added
    """
    for video in videos:
        video['identity'] = extract_identity_from_path(
            video.get('path', video['video_id']), 
            video['label']
        )
    
    # Print identity statistics
    identity_counts = defaultdict(int)
    for v in videos:
        identity_counts[v['identity']] += 1
    
    print(f"  Unique identities found: {len(identity_counts)}")
    print(f"  Videos per identity: min={min(identity_counts.values())}, "
          f"max={max(identity_counts.values())}, "
          f"mean={np.mean(list(identity_counts.values())):.1f}")
    
    return videos


def create_identity_aware_kfold_splits(videos: List[dict], n_splits: int = 5, seed: int = 42):
    """
    Create K-Fold splits that respect identity boundaries.
    
    CRITICAL FOR RESEARCH: Ensures no identity appears in both train and val.
    Uses StratifiedGroupKFold to maintain class balance while respecting groups.
    
    Args:
        videos: List of video dicts with 'identity' and 'label' fields
        n_splits: Number of folds
        seed: Random seed
    
    Returns:
        List of fold dicts with 'train' and 'val' video lists
    """
    # Assign identities if not already done
    if 'identity' not in videos[0]:
        videos = assign_identities_to_videos(videos)
    
    labels = np.array([v['label'] for v in videos])
    groups = np.array([v['identity'] for v in videos])
    indices = np.arange(len(videos))
    
    # Try StratifiedGroupKFold (maintains class balance + respects groups)
    try:
        sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
        splitter = sgkf.split(indices, labels, groups)
    except ValueError:
        # Fallback to GroupKFold if stratification fails
        print("  Warning: Using GroupKFold (stratification not possible)")
        gkf = GroupKFold(n_splits=n_splits)
        splitter = gkf.split(indices, labels, groups)
    
    folds = []
    for fold_idx, (train_idx, val_idx) in enumerate(splitter):
        train_videos = [videos[i] for i in train_idx]
        val_videos = [videos[i] for i in val_idx]
        
        # Verify NO IDENTITY LEAKAGE
        train_identities = set(v['identity'] for v in train_videos)
        val_identities = set(v['identity'] for v in val_videos)
        leaked = train_identities & val_identities
        
        if leaked:
            print(f"  ⚠ WARNING: Identity leakage detected in fold {fold_idx}: {len(leaked)} identities")
        else:
            # Also verify video IDs don't overlap
            train_ids = set(v['video_id'] for v in train_videos)
            val_ids = set(v['video_id'] for v in val_videos)
            assert len(train_ids & val_ids) == 0, f"Video ID leakage in fold {fold_idx}!"
        
        folds.append({
            'fold': fold_idx,
            'train': train_videos,
            'val': val_videos,
            'train_real': sum(1 for v in train_videos if v['label'] == 0),
            'train_fake': sum(1 for v in train_videos if v['label'] == 1),
            'val_real': sum(1 for v in val_videos if v['label'] == 0),
            'val_fake': sum(1 for v in val_videos if v['label'] == 1),
            'train_identities': len(set(v['identity'] for v in train_videos)),
            'val_identities': len(set(v['identity'] for v in val_videos)),
        })
    
    return folds


def create_random_kfold_splits(videos: List[dict], n_splits: int = 5, seed: int = 42):
    """
    Create standard K-Fold splits (random, but video-level).
    
    Use only when identity information is not available.
    WARNING: May cause identity leakage!
    """
    labels = np.array([v['label'] for v in videos])
    indices = np.arange(len(videos))
    
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    
    folds = []
    for fold_idx, (train_idx, val_idx) in enumerate(skf.split(indices, labels)):
        train_videos = [videos[i] for i in train_idx]
        val_videos = [videos[i] for i in val_idx]
        
        folds.append({
            'fold': fold_idx,
            'train': train_videos,
            'val': val_videos,
            'train_real': sum(1 for v in train_videos if v['label'] == 0),
            'train_fake': sum(1 for v in train_videos if v['label'] == 1),
            'val_real': sum(1 for v in val_videos if v['label'] == 0),
            'val_fake': sum(1 for v in val_videos if v['label'] == 1),
        })
    
    return folds


# ═══════════════════════════════════════════════════════════════════════════════
# CREATE SPLITS
# ═══════════════════════════════════════════════════════════════════════════════

print("\n" + "="*70)
print("DATA SPLIT CONFIGURATION")
print("="*70)

# Assign identities to all videos
print("\n📋 Extracting identities from video filenames...")
all_videos = assign_identities_to_videos(all_videos)

if cfg.USE_IDENTITY_SPLIT:
    print(f"\n🔒 IDENTITY-AWARE SPLITTING (Prevents Data Leakage)")
    print("-"*50)
    
    kfold_splits = create_identity_aware_kfold_splits(
        all_videos, n_splits=cfg.K_FOLDS, seed=SEED
    )
    
    # Print fold statistics
    print("\nFold Statistics (Identity-Aware):")
    for fold in kfold_splits:
        print(f"  Fold {fold['fold'] + 1}: "
              f"Train={len(fold['train'])} ({fold['train_identities']} identities), "
              f"Val={len(fold['val'])} ({fold['val_identities']} identities)")
else:
    print(f"\n⚠ RANDOM SPLITTING (May cause identity leakage!)")
    print("-"*50)
    
    kfold_splits = create_random_kfold_splits(
        all_videos, n_splits=cfg.K_FOLDS, seed=SEED
    )
    
    print("\nFold Statistics (Random):")
    for fold in kfold_splits:
        print(f"  Fold {fold['fold'] + 1}: Train={len(fold['train'])}, Val={len(fold['val'])}")

# Select current fold
if cfg.CURRENT_FOLD >= 0:
    current_split = kfold_splits[cfg.CURRENT_FOLD]
    train_videos = current_split['train']
    val_videos = current_split['val']
    
    print(f"\n✓ Using Fold {cfg.CURRENT_FOLD + 1}/{cfg.K_FOLDS}")
    print(f"  Train: {len(train_videos)} videos (Real: {current_split['train_real']}, Fake: {current_split['train_fake']})")
    print(f"  Val: {len(val_videos)} videos (Real: {current_split['val_real']}, Fake: {current_split['val_fake']})")
else:
    # Simple single split
    from sklearn.model_selection import train_test_split
    labels = [v['label'] for v in all_videos]
    train_videos, val_videos = train_test_split(
        all_videos, train_size=cfg.TRAIN_RATIO, random_state=SEED, stratify=labels
    )
    print(f"\n✓ Using single {cfg.TRAIN_RATIO*100:.0f}%-{cfg.VAL_RATIO*100:.0f}% split")

# Final verification - NO LEAKAGE
if cfg.USE_IDENTITY_SPLIT:
    train_ids = set(v['identity'] for v in train_videos)
    val_ids = set(v['identity'] for v in val_videos)
    assert len(train_ids & val_ids) == 0, "IDENTITY LEAKAGE DETECTED!"
    print(f"\n✅ NO IDENTITY LEAKAGE: {len(train_ids)} train identities, {len(val_ids)} val identities")

# Also verify video IDs
train_video_ids = set(v['video_id'] for v in train_videos)
val_video_ids = set(v['video_id'] for v in val_videos)
assert len(train_video_ids & val_video_ids) == 0, "VIDEO ID LEAKAGE DETECTED!"
print("✅ NO VIDEO LEAKAGE: Train and Val sets are completely separate")
print("="*70)



DATA SPLIT CONFIGURATION

📋 Extracting identities from video filenames...
  Unique identities found: 1042
  Videos per identity: min=1, max=9, mean=1.7

🔒 IDENTITY-AWARE SPLITTING (Prevents Data Leakage)
--------------------------------------------------

Fold Statistics (Identity-Aware):
  Fold 1: Train=1387 (828 identities), Val=367 (214 identities)
  Fold 2: Train=1397 (838 identities), Val=357 (204 identities)
  Fold 3: Train=1426 (831 identities), Val=328 (211 identities)
  Fold 4: Train=1390 (830 identities), Val=364 (212 identities)
  Fold 5: Train=1416 (841 identities), Val=338 (201 identities)

✓ Using Fold 1/5
  Train: 1387 videos (Real: 678, Fake: 709)
  Val: 367 videos (Real: 199, Fake: 168)

✅ NO IDENTITY LEAKAGE: 828 train identities, 214 val identities
✅ NO VIDEO LEAKAGE: Train and Val sets are completely separate


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# DATA AUGMENTATION + FOCAL LOSS + ALL ACCURACY ENHANCEMENTS
# ═══════════════════════════════════════════════════════════════════════════════

import cv2, torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import f1_score

# Xception uses [-1, 1] normalization
XCEPTION_MEAN = [0.5, 0.5, 0.5]
XCEPTION_STD  = [0.5, 0.5, 0.5]


class SafeToTensor(A.BasicTransform):
    def __init__(self, p=1.0):
        super().__init__(p=p)
        
    @property
    def targets(self):
        return {"image": self.apply}
        
    def apply(self, img, **params):
        if hasattr(img, 'shape') and len(img.shape) == 3:
            return torch.tensor(img.transpose(2,0,1), dtype=torch.float32)
        return torch.tensor(img, dtype=torch.float32)
        
    def get_transform_init_args_names(self):
        return ("p",)


def get_train_transforms():
    return A.Compose([
        # ── Spatial ─────────────────────────────────────────────────
        A.HorizontalFlip(p=0.5),
        A.ShiftScaleRotate(
            shift_limit=0.05, scale_limit=0.05,
            rotate_limit=10, border_mode=cv2.BORDER_REFLECT_101, p=0.3),

        # ── Color ────────────────────────────────────────────────────
        A.RandomBrightnessContrast(
            brightness_limit=0.2, contrast_limit=0.2, p=0.5),
        A.HueSaturationValue(
            hue_shift_limit=10, sat_shift_limit=20,
            val_shift_limit=15, p=0.3),
        A.RGBShift(
            r_shift_limit=15, g_shift_limit=15,
            b_shift_limit=15, p=0.3),

        # ── Compression artifacts (critical for deepfake detection) ──
        A.ImageCompression(quality_range=(40, 100), p=0.3),
        A.Downscale(scale_range=(0.5, 0.9), p=0.3),

        # ── Noise/Blur ───────────────────────────────────────────────
        A.GaussNoise(std_range=(0.02, 0.1), p=0.3),
        A.GaussianBlur(blur_limit=(3, 5), p=0.2),
        A.ISONoise(color_shift=(0.01,0.05), intensity=(0.1,0.5), p=0.2),

        # ── Occlusion ────────────────────────────────────────────────
        # Forces model to detect artifacts from multiple face regions
        A.CoarseDropout(
            max_holes=4, max_height=32, max_width=32,
            min_holes=1, min_height=8, min_width=8, fill_value=0, p=0.2),

        # ── NEW: JPEG grid artifacts simulation ──────────────────────
        # Deepfakes often show blocking in JPEG re-encoding
        A.Posterize(num_bits=4, p=0.1),

        # ── NEW: Elastic transform — simulates face warping ──────────
        A.ElasticTransform(
            alpha=120.0, sigma=12.0, p=0.15),

        # ── Normalization ────────────────────────────────────────────
        A.Normalize(mean=XCEPTION_MEAN, std=XCEPTION_STD),
        SafeToTensor(),
    ])


def get_val_transforms():
    return A.Compose([
        A.Normalize(mean=XCEPTION_MEAN, std=XCEPTION_STD),
        SafeToTensor(),
    ])


# ─── FOCAL LOSS WITH LABEL SMOOTHING ─────────────────────────────────────────
class FocalLoss(nn.Module):
    """
    IMPROVEMENT 2: Fixed focal weight computation.
    pt (well-classified probability) must be computed from ORIGINAL targets,
    not smoothed targets. Smoothed targets inflate BCE, making pt smaller,
    over-focusing on hard examples (miscalibrated focal weighting).
    """
    def __init__(self, alpha=0.5, gamma=2.0,
                 reduction='mean', label_smoothing=0.1):
        super().__init__()
        self.alpha           = alpha
        self.gamma           = gamma
        self.reduction       = reduction
        self.label_smoothing = label_smoothing

    def forward(self, inputs, targets):
        # Step 1: Compute pt from ORIGINAL targets (correct focal weight)
        p   = torch.sigmoid(inputs)
        pt  = targets * p + (1 - targets) * (1 - p)  # well-classified prob
        pt  = torch.clamp(pt, min=1e-7, max=1.0 - 1e-7)  # FIX: Numerical stability
        focal_weight = (1 - pt) ** self.gamma
        alpha_t = self.alpha * targets + (1 - self.alpha) * (1 - targets)

        # Step 2: Apply label smoothing ONLY to BCE loss (not to focal weight)
        targets_smooth = (targets * (1 - self.label_smoothing)
                          + 0.5 * self.label_smoothing)
        bce = F.binary_cross_entropy_with_logits(
            inputs, targets_smooth, reduction='none')

        loss = alpha_t * focal_weight * bce
        if self.reduction == 'mean':  return torch.mean(loss)
        if self.reduction == 'sum':   return torch.sum(loss)
        return loss


# ─── MIXUP ───────────────────────────────────────────────────────────────────
def mixup_data(frames, labels, alpha=0.2):
    """
    ISSUE 3 FIX: Force non-identity permutation.
    With BATCH_SIZE=2, randperm returns [0,1] 50% of time (no mixing).
    Roll by 1 guarantees actual mixing every time.
    """
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    B = frames.size(0)
    if B < 2:
        return frames, labels, labels, 1.0
    # Force non-identity permutation
    idx = torch.roll(torch.arange(B, device=frames.device), shifts=1)
    mixed = lam * frames + (1 - lam) * frames[idx]
    return mixed, labels, labels[idx], lam



# ─── CUTMIX (IMPROVEMENT 4) ─────────────────────────────────────────────────
def cutmix_data(frames, labels, alpha=1.0):
    """
    CutMix: paste a random rectangular patch from one video onto another.
    Forces detection of local manipulation artifacts rather than global statistics.
    Proven superior to Mixup for deepfake detection (local artifacts are key).
    """
    lam = np.random.beta(alpha, alpha)
    B, T, C, H, W = frames.shape
    # Force non-identity: always swap between the samples
    idx = torch.roll(torch.arange(B, device=frames.device), shifts=1)

    # Generate random bounding box
    cut_ratio = np.sqrt(1.0 - lam)
    cut_h = int(H * cut_ratio)
    cut_w = int(W * cut_ratio)
    cx = np.random.randint(W)
    cy = np.random.randint(H)
    x1 = np.clip(cx - cut_w // 2, 0, W)
    x2 = np.clip(cx + cut_w // 2, 0, W)
    y1 = np.clip(cy - cut_h // 2, 0, H)
    y2 = np.clip(cy + cut_h // 2, 0, H)

    mixed = frames.clone()
    mixed[:, :, :, y1:y2, x1:x2] = frames[idx, :, :, y1:y2, x1:x2]

    # Adjust lambda to actual cut area
    lam = 1.0 - (x2 - x1) * (y2 - y1) / (W * H)
    return mixed, labels, labels[idx], lam

def mixup_criterion(criterion, pred, ya, yb, lam):
    return lam * criterion(pred, ya) + (1 - lam) * criterion(pred, yb)


# ─── SAM OPTIMIZER ───────────────────────────────────────────────────────────
class SAM(torch.optim.Optimizer):
    """Sharpness Aware Minimization — finds flatter, more general minima."""
    def __init__(self, params, base_optimizer=torch.optim.AdamW,
                 rho=0.05, **kwargs):
        defaults = dict(rho=rho, **kwargs)
        super().__init__(params, defaults)
        self.base_optimizer = base_optimizer(self.param_groups, **kwargs)
        self.param_groups   = self.base_optimizer.param_groups

    @torch.no_grad()
    def first_step(self, zero_grad=False):
        norm = self._grad_norm()
        for group in self.param_groups:
            scale = group['rho'] / (norm + 1e-12)
            for p in group['params']:
                if p.grad is None: continue
                e_w = p.grad * scale.to(p)
                p.add_(e_w)
                self.state[p]['e_w'] = e_w
        if zero_grad: self.zero_grad()

    @torch.no_grad()
    def second_step(self, zero_grad=False):
        for group in self.param_groups:
            for p in group['params']:
                if p.grad is None: continue
                p.sub_(self.state[p]['e_w'])
        self.base_optimizer.step()
        if zero_grad: self.zero_grad()

    def _grad_norm(self):
        dev = self.param_groups[0]['params'][0].device
        return torch.norm(torch.stack([
            p.grad.norm(p=2).to(dev)
            for g in self.param_groups for p in g['params']
            if p.grad is not None]), p=2)

    def step(self, closure=None):
        raise NotImplementedError("SAM: use first_step/second_step manually")


# ─── HARD NEGATIVE SAMPLER ───────────────────────────────────────────────────
def create_hard_negative_loader(model, dataset, device, batch_size=2):
    """
    Re-weight sampler to focus on uncertain examples.
    Examples near 0.5 threshold are hardest — most informative.
    
    ISSUE 1 FIX: Uses try/finally to guarantee transform restoration
    even if an exception occurs during scoring.
    """
    model.eval()
    all_probs = []
    original_transform = dataset.transform
    
    try:
        # Issue 13 FIX: Use val transforms for scoring
        dataset.transform = get_val_transforms()
        tmp = DataLoader(dataset, batch_size=cfg.BATCH_SIZE, shuffle=False, num_workers=0)
        with torch.no_grad():
            for batch in tmp:
                frames = batch['frames'].to(device)
                mask   = batch['mask'].to(device)
                logits = model(frames, mask).squeeze()
                probs  = torch.sigmoid(logits).cpu().tolist()
                if not isinstance(probs, list): probs = [probs]
                all_probs.extend(probs)
    finally:
        # ALWAYS restore training transforms, even on exception
        dataset.transform = original_transform
        del tmp  # FIX: Free DataLoader to prevent memory leak
        gc.collect()
    
    model.train()

    # IMPROVEMENT 9: Class-balanced hard negative mining
    # Without per-class normalization, uncertain real videos (minority class)
    # get oversampled, undermining FocalLoss class weighting
    all_labels_for_mining = [v['label'] for v in dataset.videos]
    hardness = np.array([1.0 - abs(p - 0.5) * 2 for p in all_probs])

    # Normalize within each class separately so both classes contribute equally
    hardness_balanced = np.zeros_like(hardness)
    for cls in [0, 1]:
        mask_cls = np.array(all_labels_for_mining) == cls
        if mask_cls.sum() > 0:
            h_cls = hardness[mask_cls]
            h_cls = h_cls / (h_cls.sum() + 1e-12)
            hardness_balanced[mask_cls] = h_cls * 0.5  # each class = 50% of total weight
    hardness = hardness_balanced

    sampler = torch.utils.data.WeightedRandomSampler(
        weights=hardness, num_samples=len(dataset), replacement=True)
    return DataLoader(dataset, batch_size=batch_size,
                      sampler=sampler, num_workers=0,
                      pin_memory=False, drop_last=True)


# ─── OPTIMAL THRESHOLD ───────────────────────────────────────────────────────
def find_optimal_threshold(y_true, y_prob):
    best_f1, best_thresh = 0, 0.5
    y_t = list(y_true) if hasattr(y_true,'__iter__') else [y_true]
    y_p = list(y_prob) if hasattr(y_prob,'__iter__') else [y_prob]
    for t in [x/100 for x in range(5, 96, 1)]:
        preds = [1 if p >= t else 0 for p in y_p]
        f1 = f1_score(y_t, preds, zero_division=0)
        if f1 > best_f1:
            best_f1, best_thresh = f1, t
    return best_thresh, best_f1


def calculate_class_weights(labels):
    ll = list(labels)
    n  = len(ll)
    n0 = sum(1 for l in ll if l == 0)
    n1 = sum(1 for l in ll if l == 1)
    return {0: n/(2*n0) if n0 > 0 else 1.0,
            1: n/(2*n1) if n1 > 0 else 1.0}


def compute_eer(label, pred):
    """Computes Equal Error Rate."""
    if len(np.unique(label)) < 2:
        return 0.5
    fpr, tpr, thresholds = roc_curve(label, pred)
    fnr = 1 - tpr
    eer = fpr[np.nanargmin(np.absolute(fnr - fpr))]
    return eer


print("✓ All augmentations, losses, and optimizers defined")
print(f"  Backbone norm   : Xception [-1,1]")
print(f"  Focal + smooth  : {cfg.LABEL_SMOOTHING}")
print(f"  SAM optimizer   : rho=0.05")
print(f"  Mixup           : alpha={cfg.MIXUP_ALPHA}")
print(f"  Hard mining     : epoch {cfg.HARD_MINING_EPOCH}+")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# PYTORCH DATASETS — DISK-LOADING VERSION (P100 RAM-SAFE)
# Loads face data from .npy cache files on demand.
# RAM stays flat regardless of dataset size.
# ═══════════════════════════════════════════════════════════════════════

class DeepfakeVideoDataset(Dataset):
    """
    Video-level dataset that loads faces from disk per __getitem__.
    RAM usage = ~1 video worth of faces at any moment.
    """

    def __init__(self, videos: List[dict], cache_index: dict,
                 transform=None, max_frames: int = 20,
                 is_train: bool = True):
        self.transform   = transform
        self.max_frames  = max_frames
        self.is_train    = is_train

        self.videos = []
        for video in videos:
            vid_id = video['video_id']
            if vid_id in cache_index:
                self.videos.append({
                    'video_id'   : vid_id,
                    'label'      : video['label'],
                    'cache_path' : cache_index[vid_id],
                    'source'     : video.get('source', 'unknown')
                })

        print(f"  Dataset: {len(self.videos)} videos "
              f"({'train' if is_train else 'val'})")

    def __len__(self):
        return len(self.videos)

    def __getitem__(self, idx):
        video      = self.videos[idx]
        label      = video['label']
        video_id   = video['video_id']

        # Load from disk — only this video's faces in RAM
        faces = np.load(video['cache_path'])  # (T, H, W, 3) uint8

        # Sample or pad to max_frames
        # IMPROVEMENT 10: Add random jitter during training for frame diversity
        n = len(faces)
        if n >= self.max_frames:
            step = n / self.max_frames
            # During training: add random jitter up to ± half step for variety across epochs
            if self.is_train:
                jitters = np.random.uniform(-step * 0.4, step * 0.4, size=self.max_frames)
            else:
                jitters = np.zeros(self.max_frames)
            indices = [int(np.clip(i * step + jitters[i], 0, n - 1))
                       for i in range(self.max_frames)]
        else:
            indices = list(range(n))
            while len(indices) < self.max_frames:
                indices.append(n - 1)

        selected = [faces[i].astype('uint8') for i in indices]

        # Apply transforms frame by frame
        frame_tensors = []
        for face in selected:
            if self.transform:
                frame_tensors.append(
                    self.transform(image=face)['image'])
            else:
                t = torch.tensor(
                    face.transpose(2, 0, 1),
                    dtype=torch.float32) / 255.0
                frame_tensors.append(t)

        frames = torch.stack(frame_tensors)         # (T, C, H, W)
        # FIX: Proper padding mask - True for actual frames, False for padding
        actual_frames = min(n, self.max_frames)
        mask = torch.zeros(self.max_frames, dtype=torch.bool)
        mask[:actual_frames] = True

        # Free faces array immediately
        del faces

        return {
            'frames'  : frames,
            'label'   : torch.tensor(label, dtype=torch.float32),
            'video_id': video_id,
            'mask'    : mask
        }


# ── CREATE DATASETS ──────────────────────────────────────────────────
print("\nCreating VIDEO-LEVEL datasets (disk-loading)...")
train_dataset = DeepfakeVideoDataset(
    train_videos, cache_index,
    transform=get_train_transforms(),
    max_frames=cfg.FRAMES_PER_VIDEO,
    is_train=True
)
val_dataset = DeepfakeVideoDataset(
    val_videos, cache_index,
    transform=get_val_transforms(),
    max_frames=cfg.FRAMES_PER_VIDEO,
    is_train=False
)

train_labels  = [v['label'] for v in train_videos]
class_weights = calculate_class_weights(train_labels)
print(f"\n  Class weights: {class_weights}")

train_loader = DataLoader(
    train_dataset,
    batch_size=cfg.BATCH_SIZE,
    shuffle=True,
    num_workers=0,      # CRITICAL: Must be 0 on P100
    pin_memory=False,  # Issue 33 FIX: pin_memory with num_workers=0 is useless
    drop_last=True,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=cfg.BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=False,  # Issue 33 FIX: pin_memory with num_workers=0 is useless
)

print(f"\n✓ Train loader: {len(train_loader)} batches")
print(f"✓ Val loader  : {len(val_loader)} batches")
print(f"✓ RAM-safe    : faces loaded from disk per batch")


Creating VIDEO-LEVEL datasets (disk-loading)...
  Dataset: 1386 videos (train)
  Dataset: 367 videos (val)

  Class weights: {0: 1.0228613569321534, 1: 0.9781382228490832}

✓ Train loader: 693 batches
✓ Val loader  : 184 batches
✓ RAM-safe    : faces loaded from disk per batch


## 4. CNN Model Architecture

In [30]:
# ═══════════════════════════════════════════════════════════════════════════════
# SPATIO-TEMPORAL CNN MODEL (P100 COMPATIBLE - STRICT FP32)
# ═══════════════════════════════════════════════════════════════════════════════

# CRITICAL: Clear GPU memory from face extraction (MTCNN)
import gc
import torch

# Delete MTCNN from GPU — it held ~15GB during extraction
if 'face_extractor' in dir():
    del face_extractor
gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()

# Confirm memory is free before loading model
if torch.cuda.is_available():
    free = torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated(0)
    print(f"✓ GPU memory freed: {free / 1e9:.1f} GB available")

# ... rest of your model code unchanged

✓ GPU memory freed: 16.6 GB available


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# SPATIO-TEMPORAL CNN MODEL WITH FREQUENCY BRANCH (P100 COMPATIBLE)
# ═══════════════════════════════════════════════════════════════════════════════

class FrequencyBranch(nn.Module):
    """
    FFT frequency domain branch.
    GAN-generated deepfakes leave spectral artifacts invisible
    spatially but detectable as frequency peaks. Used in F3Net,
    SPSL, and other published deepfake detection papers.
    """
    def __init__(self, out_dim=128):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.Dropout2d(0.1),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d(16),
            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.Dropout2d(0.1),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d(4),
        )
        self.fc = nn.Sequential(
            nn.Linear(64 * 4 * 4, out_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3)
        )

    def forward(self, x):
        # x: (N, C, H, W)
        gray = x.mean(dim=1, keepdim=True)         # grayscale
        # ISSUE 5 FIX: Downsample to 64x64 before FFT for speed
        gray_small = F.adaptive_avg_pool2d(gray, (64, 64))
        fft  = torch.fft.fft2(gray_small)
        mag  = torch.abs(fft)
        mag  = torch.log(mag + 1e-8)               # log scale
        # Shift DC to center
        mag  = torch.roll(mag, shifts=(mag.shape[-2]//2,
                                       mag.shape[-1]//2),
                          dims=(-2, -1))
        # IMPROVEMENT 6: Zero DC component — it carries no artifact information
        # and overwhelms the subtle frequency peaks from deepfake generation
        cy, cx = mag.shape[-2] // 2, mag.shape[-1] // 2
        mag = mag.clone()
        mag[:, :, cy-1:cy+2, cx-1:cx+2] = 0.0   # zero 3×3 DC region
        
        feat = self.conv(mag)
        feat = feat.view(feat.size(0), -1)
        return self.fc(feat)


class TemporalAttention(nn.Module):
    def __init__(self, feature_dim: int, num_heads: int = 4,
                 dropout: float = 0.1):
        super().__init__()
        self.attention  = nn.MultiheadAttention(
            embed_dim=feature_dim, num_heads=num_heads,
            dropout=dropout, batch_first=True)
        self.layer_norm = nn.LayerNorm(feature_dim)
        self.dropout    = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        kpm      = ~mask if mask is not None else None
        attn_out, attn_w = self.attention(x, x, x,
                                          key_padding_mask=kpm)
        x = self.layer_norm(x + self.dropout(attn_out))
        if mask is not None:
            me     = mask.unsqueeze(-1).float()
            pooled = (x * me).sum(1) / me.sum(1).clamp(min=1)
        else:
            pooled = x.mean(dim=1)
        return pooled, attn_w


class SpatioTemporalDeepfakeCNN(nn.Module):
    def __init__(self, model_name='xception', hidden_dim=256,
                 dropout=0.3, pretrained=True,
                 temporal_type='bilstm_attention',
                 lstm_hidden=256, lstm_layers=2,
                 attention_heads=4, freeze_backbone=False,
                 use_gradient_checkpointing=False,  # ISSUE 2 FIX: Safe at BATCH_SIZE=2, avoids cudnn interaction on P100 backward
                 use_freq_branch=True):    # NEW
        super().__init__()
        self.temporal_type      = temporal_type
        self.use_gradient_checkpointing = use_gradient_checkpointing  # ISSUE 10 FIX: Honor parameter
        self.use_freq_branch    = use_freq_branch

        # ── Spatial backbone ─────────────────────────────────────────
        self.backbone     = timm.create_model(
            model_name, pretrained=pretrained,
            num_classes=0, global_pool='avg')
        self.backbone_dim = self.backbone.num_features

        # IMPROVEMENT 1: Enable gradient checkpointing to save VRAM
        if use_gradient_checkpointing:
            self.backbone.set_grad_checkpointing(enable=True)
            print("  Gradient checkpointing ENABLED (trades compute for VRAM)")

        if freeze_backbone:
            for p in self.backbone.parameters():
                p.requires_grad = False

        # ── Frequency branch (NEW) ───────────────────────────────────
        self.freq_dim    = 128
        self.freq_branch = FrequencyBranch(out_dim=self.freq_dim)
        combined_dim     = self.backbone_dim + self.freq_dim

        # ── IMPROVEMENT 5: Projection layer (2176 → 512) ────────────────────────
        # Reduces 8.5× compression to 4.3× — LSTM learns temporal patterns easier
        PROJ_DIM = 512
        self.input_proj = nn.Sequential(
            nn.Linear(combined_dim, PROJ_DIM),
            nn.LayerNorm(PROJ_DIM),
            nn.GELU(),
            nn.Dropout(dropout * 0.5)
        )
        lstm_input_dim = PROJ_DIM

        # ── Temporal ─────────────────────────────────────────────────
        if temporal_type in ['bilstm', 'bilstm_attention']:
            self.temporal = nn.LSTM(
                input_size=lstm_input_dim,  # IMPROVEMENT 5: was combined_dim
                hidden_size=lstm_hidden, num_layers=lstm_layers,
                batch_first=True, bidirectional=True,
                dropout=dropout if lstm_layers > 1 else 0)
            # Issue 30 FIX: Removed flatten_parameters monkeypatch (dead code with cudnn disabled)
            temporal_out_dim = lstm_hidden * 2
            
            # IMPROVEMENT 7 & ISSUE 1 FIX: Sinusoidal positional encoding
            # Attention treats frames as unordered — add temporal position info
            assert temporal_out_dim == lstm_hidden * 2, "temporal_out_dim must equal lstm_hidden * 2 for correct pos_enc"
            max_len = 50  # more than FRAMES_PER_VIDEO
            pe = torch.zeros(max_len, temporal_out_dim)
            position = torch.arange(max_len).unsqueeze(1).float()
            div_term = torch.exp(torch.arange(0, temporal_out_dim, 2).float()
                                 * (-math.log(10000.0) / temporal_out_dim))
            pe[:, 0::2] = torch.sin(position * div_term)
            pe[:, 1::2] = torch.cos(position * div_term)
            self.register_buffer('pos_enc', pe.unsqueeze(0))
            assert cfg.FRAMES_PER_VIDEO <= 50, f"FRAMES_PER_VIDEO ({cfg.FRAMES_PER_VIDEO}) exceeds max positional encoding length (50)"

            if temporal_type == 'bilstm_attention':
                self.temporal_attention = TemporalAttention(
                    feature_dim=temporal_out_dim,
                    num_heads=attention_heads, dropout=dropout)
        elif temporal_type == 'transformer':
            encoder_layer = nn.TransformerEncoderLayer(
                d_model=combined_dim, nhead=attention_heads,
                dim_feedforward=combined_dim * 2,
                dropout=dropout, activation='gelu', batch_first=True)
            self.temporal = nn.TransformerEncoder(
                encoder_layer, num_layers=lstm_layers)
            self.temporal_attention = TemporalAttention(
                feature_dim=combined_dim,
                num_heads=attention_heads, dropout=dropout)
            temporal_out_dim = combined_dim
        else:
            raise ValueError(f"Unknown: {temporal_type}")

        self.temporal_out_dim = temporal_out_dim

        # ── Classifier ───────────────────────────────────────────────
        # CRITICAL BUG 1 FIX: Replace BatchNorm1d with LayerNorm
        # BatchNorm1d with BATCH_SIZE=2 creates unstable variance estimates
        # LayerNorm normalizes over features, not batch (batch-size agnostic)
        self.classifier = nn.Sequential(
            nn.Linear(temporal_out_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),  # FIXED: was BatchNorm1d
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.LayerNorm(hidden_dim // 2),  # FIXED: was BatchNorm1d
            nn.GELU(),
            nn.Dropout(dropout / 2),
            nn.Linear(hidden_dim // 2, 1)
        )
        self._init_weights()
        print(f"\n✓ SpatioTemporalDeepfakeCNN")
        print(f"  Backbone   : {model_name} ({self.backbone_dim}d)")
        print(f"  Freq branch: {self.freq_dim}d FFT features")
        print(f"  Combined   : {combined_dim}d → BiLSTM({lstm_hidden}×2)")
        print(f"  ⚠️ P100: FP32, cuDNN bypass, no checkpointing")

    def _init_weights(self):
        for m in self.classifier.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def extract_frame_features(self, frames):
        B, T, C, H, W = frames.shape
        flat = frames.view(B * T, C, H, W)
        spatial = self.backbone(flat)                 # (B*T, backbone_dim)
        freq    = self.freq_branch(flat)              # (B*T, freq_dim)
        combined = torch.cat([spatial, freq], dim=1)  # (B*T, 2176)
        projected = self.input_proj(combined)          # IMPROVEMENT 5: (B*T, 512)
        return projected.view(B, T, -1)

    def forward(self, frames, mask=None):
        features = self.extract_frame_features(frames)
        if self.temporal_type == 'bilstm':
            with torch.backends.cudnn.flags(enabled=False):
                lstm_out, _ = self.temporal(features)
            if mask is not None:
                me     = mask.unsqueeze(-1).float()
                pooled = (lstm_out * me).sum(1) / me.sum(1).clamp(min=1)
            else:
                pooled = lstm_out.mean(dim=1)
        elif self.temporal_type == 'bilstm_attention':
            with torch.backends.cudnn.flags(enabled=False):
                lstm_out, _ = self.temporal(features)
            T = lstm_out.size(1)
            lstm_out = lstm_out + self.pos_enc[:, :T, :]  # IMPROVEMENT 7
            pooled, _ = self.temporal_attention(lstm_out, mask)
        elif self.temporal_type == 'transformer':
            attn_mask = ~mask if mask is not None else None
            out    = self.temporal(features,
                                   src_key_padding_mask=attn_mask)
            pooled, _ = self.temporal_attention(out, mask)
        return self.classifier(pooled).squeeze(-1)

    def get_attention_weights(self, frames, mask=None):
        with torch.no_grad():
            features = self.extract_frame_features(frames)
            if self.temporal_type == 'bilstm_attention':
                with torch.backends.cudnn.flags(enabled=False):
                    lstm_out, _ = self.temporal(features)
                T = lstm_out.size(1)
                lstm_out = lstm_out + self.pos_enc[:, :T, :]
                _, w = self.temporal_attention(lstm_out, mask)
                return w
        return None


model = SpatioTemporalDeepfakeCNN(
    model_name=cfg.MODEL_NAME,
    hidden_dim=cfg.HIDDEN_DIM,
    dropout=cfg.DROPOUT,
    pretrained=True,
    temporal_type=cfg.TEMPORAL_TYPE,
    lstm_hidden=cfg.LSTM_HIDDEN,
    lstm_layers=cfg.LSTM_LAYERS,
    attention_heads=cfg.ATTENTION_HEADS,
    freeze_backbone=cfg.FREEZE_BACKBONE,
    use_gradient_checkpointing=False,  # ISSUE 2 FIX: Safe at BATCH_SIZE=2, avoids cudnn interaction on P100 backward
    use_freq_branch=True
)
model = model.to(DEVICE)
print(f"✓ Model on {DEVICE}")
total_params = sum(p.numel() for p in model.parameters())
trainable    = sum(p.numel() for p in model.parameters()
                   if p.requires_grad)
print(f"  Total params    : {total_params:,}")
print(f"  Trainable params: {trainable:,}")

## 5. Training Protocol

In [32]:
# ═══════════════════════════════════════════════════════════════════════════════
# METRICS & TRAINING FUNCTIONS (CRASH-PROOF P100 VERSION)
# ═══════════════════════════════════════════════════════════════════════════════

# Bug 1 FIX: FocalLoss is defined in Cell 19 with label_smoothing - DO NOT redefine here
# Issue 36 FIX: compute_eer is defined in Cell 19 - DO NOT redefine here

# Issue 17 FIX: Removed train_one_epoch_with_accumulation (dead code, never called)

def validate(model, loader, criterion, device):
    """
    Validation loop with crash-proof NumPy bypass.
    """
    model.eval()
    total_loss = 0.0
    all_preds, all_labels = [], []
    num_samples = 0

    pbar = tqdm(loader, desc="Validation", leave=False)

    with torch.no_grad():
        for batch in pbar:
            frames = batch['frames'].to(device)
            labels = batch['label'].to(device)
            mask = batch.get('mask', None)
            if mask is not None:
                mask = mask.to(device)

            batch_size = labels.size(0)

            # Forward pass
            logits = model(frames, mask).squeeze()
            if logits.dim() == 0:
                logits = logits.unsqueeze(0)

            loss = criterion(logits, labels.float())

            total_loss += loss.item() * batch_size
            num_samples += batch_size

            # Safe conversion
            probs = torch.sigmoid(logits).cpu().tolist()
            if not isinstance(probs, list):
                probs = [probs]
            all_preds.extend(probs)

            labels_list = labels.cpu().tolist()
            if not isinstance(labels_list, list):
                labels_list = [labels_list]
            all_labels.extend(labels_list)

            pbar.set_postfix({'loss': f"{total_loss/num_samples:.4f}"})

    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    binary_preds = (all_preds > 0.5).astype(int)

    epoch_loss = total_loss / max(1, num_samples)
    epoch_acc = accuracy_score(all_labels, binary_preds) if len(all_labels) > 0 else 0
    epoch_auc = roc_auc_score(all_labels, all_preds) if len(np.unique(all_labels)) > 1 else 0.5
    epoch_eer = compute_eer(all_labels, all_preds) if len(np.unique(all_labels)) > 1 else 0.5

    epoch_f1 = f1_score(all_labels, binary_preds, zero_division=0) if len(all_labels) > 0 else 0
    epoch_precision = precision_score(all_labels, binary_preds, zero_division=0) if len(all_labels) > 0 else 0
    epoch_recall = recall_score(all_labels, binary_preds, zero_division=0) if len(all_labels) > 0 else 0

    return {
        'loss': epoch_loss, 'acc': epoch_acc, 'auc': epoch_auc, 'eer': epoch_eer,
        'f1': epoch_f1, 'precision': epoch_precision, 'recall': epoch_recall
    }

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# TRAINING LOOP - SAM + SWA + HARD MINING + PROGRESSIVE FRAMES
# ═══════════════════════════════════════════════════════════════════════════════

criterion = FocalLoss(
    alpha=cfg.FOCAL_ALPHA, gamma=cfg.FOCAL_GAMMA,
    label_smoothing=cfg.LABEL_SMOOTHING)

param_groups = [
    {'params': model.backbone.parameters(),
     'lr': cfg.LEARNING_RATE / 10},
    {'params': model.input_proj.parameters(),  # BUG 3 FIX: Add missing input_proj
     'lr': cfg.LEARNING_RATE},
    {'params': model.freq_branch.parameters(),
     'lr': cfg.LEARNING_RATE},
    {'params': model.temporal.parameters(),
     'lr': cfg.LEARNING_RATE},
    {'params': model.classifier.parameters(),
     'lr': cfg.LEARNING_RATE},
]
if hasattr(model, 'temporal_attention'):
    param_groups.insert(3, {
        'params': model.temporal_attention.parameters(),
        'lr': cfg.LEARNING_RATE})

optimizer = SAM(param_groups,
                base_optimizer=torch.optim.AdamW,
                rho=0.05,
                weight_decay=cfg.WEIGHT_DECAY)

import math
steps_per_epoch      = math.ceil(len(train_loader) /
                                  cfg.GRAD_ACCUMULATION_STEPS)
# CRITICAL BUG 3 FIX: Scheduler should decay to min LR exactly when SWA starts
# Otherwise LR is still high when abruptly switched to SWA_LR
scheduler_epochs     = cfg.SWA_START if cfg.USE_SWA else cfg.NUM_EPOCHS
total_training_steps = scheduler_epochs * steps_per_epoch
warmup_steps         = int(cfg.WARMUP_RATIO * total_training_steps)

# Issue 7 FIX: Implement actual warmup with linear warmup + cosine decay
# ISSUE 4 FIX: LambdaLR step counter implicitly tracks batches since we call base_scheduler.step() per batch, not per epoch.
def get_lr_lambda(current_step, warmup_steps, total_steps):
    if current_step < warmup_steps:
        return float(current_step) / float(max(1, warmup_steps))
    progress = float(current_step - warmup_steps) / float(max(1, total_steps - warmup_steps))
    return max(0.0, 0.5 * (1.0 + math.cos(math.pi * progress)))

base_scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer.base_optimizer,
    lr_lambda=lambda step: get_lr_lambda(step, warmup_steps, total_training_steps)
)

# SWA setup
swa_model = AveragedModel(model)
swa_scheduler = SWALR(
    optimizer.base_optimizer,
    swa_lr=cfg.SWA_LR, anneal_epochs=5)

history = {
    'train_loss':[], 'train_acc':[], 'train_auc':[], 'train_eer':[],
    'val_loss':[],   'val_acc':[],   'val_auc':[],   'val_eer':[],
    'val_f1':[], 'val_precision':[], 'val_recall':[], 'lr':[]
}

best_val_auc     = 0.0
best_val_eer     = 1.0
best_epoch       = 0
patience         = cfg.PATIENCE
patience_counter = 0
current_train_loader = train_loader   # may swap to hard-mining loader
original_train_loader = train_loader  # Issue 11 FIX: Keep reference for SWA BN update

# Bug 2 FIX: Initialize current_frames before the epoch loop
current_frames = cfg.FRAMES_PER_VIDEO

# ── GRADUAL UNFREEZING: freeze backbone initially ────────────────────
# ISSUE 8 FIX: Honor cfg.FREEZE_BACKBONE setting
if cfg.FREEZE_BACKBONE or cfg.UNFREEZE_EPOCH > 0:
    for p in model.backbone.parameters():
        p.requires_grad = False
    print("Backbone frozen — temporal layers learn first")
else:
    print("Backbone trainable from start (FREEZE_BACKBONE=False, UNFREEZE_EPOCH=0)")

print("\n" + "="*70)
print("TRAINING — ALL ENHANCEMENTS ACTIVE")
print("="*70)
print(f"  SAM optimizer, Mixup, Hard mining, SWA, Progressive frames")
print(f"  Gradual unfreeze @ epoch {cfg.UNFREEZE_EPOCH}")
print(f"  Hard negative mining @ epoch {cfg.HARD_MINING_EPOCH}")
print(f"  SWA starts @ epoch {cfg.SWA_START}")
print(f"  Warmup steps: {warmup_steps}")
print("="*70 + "\n")

backbone_warmup_until = cfg.UNFREEZE_EPOCH + 2  # keep backbone frozen for 2 epochs after unfreeze

for epoch in range(cfg.NUM_EPOCHS):

    # ── Progressive frames curriculum (TRAIN ONLY) ───────────────────
    if cfg.USE_PROGRESSIVE_FRAMES:
        if epoch < 5:
            current_frames = 5
        elif epoch < 15:
            current_frames = 10
        else:
            current_frames = cfg.FRAMES_PER_VIDEO
        if hasattr(train_dataset, 'max_frames'):
            train_dataset.max_frames = current_frames
        # Issue 6 FIX: DO NOT update val_dataset.max_frames
        # Validation must use full frames for consistent AUC comparison

    # ── Gradual unfreezing ───────────────────────────────────────────
    if epoch == cfg.UNFREEZE_EPOCH:
        for p in model.backbone.parameters():
            p.requires_grad = True
        # BUG 4 & ISSUE 8 FIX: Backbone AdamW warmup prevents scheduler override
        # Note for publication: Backbone LR is heavily reduced for 2 epochs post-unfreeze 
        # to preserve learned temporal representations while frozen momentum estimates warm up.
        optimizer.base_optimizer.param_groups[0]['lr'] = cfg.LEARNING_RATE / 100
        backbone_warmup_until = epoch + 2
        print(f"\n  Epoch {epoch+1}: Backbone UNFROZEN with warmup LR={cfg.LEARNING_RATE/100:.1e}")

    # ── Hard negative mining ─────────────────────────────────────────
    if epoch == cfg.HARD_MINING_EPOCH:
        print(f"\n  Epoch {epoch+1}: Switching to hard negative mining")
        current_train_loader = create_hard_negative_loader(
            model, train_dataset, DEVICE, cfg.BATCH_SIZE)
        # MINOR 4 FIX: Clear cache after heavy evaluation phase
        gc.collect()
        torch.cuda.empty_cache()

    print(f"\nEpoch {epoch+1}/{cfg.NUM_EPOCHS}"
          f" [frames={current_frames}]")
    print("-" * 50)

    # ── TRAINING EPOCH ───────────────────────────────────────────────
    model.train()
    total_loss = 0.0
    all_preds, all_labels = [], []
    num_samples  = 0
    current_lr   = optimizer.base_optimizer.param_groups[1]['lr']
    optimizer.base_optimizer.zero_grad()

    pbar = tqdm(current_train_loader, desc="Training", leave=False)
    for batch_idx, batch in enumerate(pbar):
        frames = batch['frames'].to(DEVICE)
        labels = batch['label'].to(DEVICE)
        mask   = batch.get('mask', None)
        if mask is not None:
            mask = mask.to(DEVICE)
        batch_size = labels.size(0)

        # ── Mixup ────────────────────────────────────────────────────
        # ── Mixup / CutMix (IMPROVEMENT 4) ──────────────────────────────────────
        use_mixup_this_batch = (cfg.USE_MIXUP
                                and np.random.random() > 0.5
                                and epoch >= 0) # IMPROVEMENT 1 (Research Grade): Starts mixup from epoch 0
        if use_mixup_this_batch:
            # Alternate: 50% Mixup, 50% CutMix
            # CutMix superior for local artifact detection (deepfake boundaries)
            if np.random.random() > 0.5:
                frames, ya, yb, lam = mixup_data(frames, labels, cfg.MIXUP_ALPHA)
            else:
                frames, ya, yb, lam = cutmix_data(frames, labels, alpha=1.0)

        # ISSUE 2 FIX: SAM requires identical data for both steps
        # Gradient accumulation enabled (GRAD_ACCUMULATION_STEPS=4)
        logits = model(frames, mask).squeeze()
        if logits.dim() == 0: logits = logits.unsqueeze(0)

        if use_mixup_this_batch:
            loss = mixup_criterion(criterion, logits, ya, yb, lam)
        else:
            loss = criterion(logits, labels.float())

        # Issue 4 FIX: Do NOT divide first step loss for SAM
        loss.backward()

        if ((batch_idx + 1) % cfg.GRAD_ACCUMULATION_STEPS == 0 or
                batch_idx + 1 == len(current_train_loader)):
            torch.nn.utils.clip_grad_norm_(
                model.parameters(), max_norm=1.0)
            optimizer.first_step(zero_grad=True)

            # SAM second step - use same batch data
            logits2 = model(frames, mask).squeeze()
            if logits2.dim() == 0: logits2 = logits2.unsqueeze(0)
            if use_mixup_this_batch:
                loss2 = mixup_criterion(
                    criterion, logits2, ya, yb, lam)
            else:
                loss2 = criterion(logits2, labels.float())
            # Issue 4 FIX: Do NOT divide loss2 - SAM needs same scale
            loss2.backward()
            torch.nn.utils.clip_grad_norm_(
                model.parameters(), max_norm=1.0)
            optimizer.second_step(zero_grad=True)

            # Update LR scheduler
            # ISSUE 8 FIX: skip scheduler step for 2 epochs after unfreeze to preserve warmup LR
            if epoch < cfg.SWA_START and not (epoch >= cfg.UNFREEZE_EPOCH and epoch < backbone_warmup_until):
                base_scheduler.step()
            current_lr = (optimizer.base_optimizer
                          .param_groups[1]['lr'])

        total_loss  += loss.item() * batch_size
        num_samples += batch_size

        probs = torch.sigmoid(logits).detach().cpu().tolist()
        if not isinstance(probs, list): probs = [probs]
        all_preds.extend(probs)
        labs = labels.cpu().tolist()
        if not isinstance(labs, list): labs = [labs]
        all_labels.extend(labs)

        pbar.set_postfix({'loss': f"{total_loss/num_samples:.4f}",
                          'lr':   f"{current_lr:.1e}"})

    all_preds  = np.array(all_preds)
    all_labels = np.array(all_labels)
    train_loss = total_loss / max(1, num_samples)
    train_acc  = accuracy_score(
        all_labels, (all_preds > 0.5).astype(int))
    train_auc  = (roc_auc_score(all_labels, all_preds)
                  if len(np.unique(all_labels)) > 1 else 0.5)
    train_eer  = (compute_eer(all_labels, all_preds)
                  if len(np.unique(all_labels)) > 1 else 0.5)

    # ── SWA accumulation ─────────────────────────────────────────────
    if epoch >= cfg.SWA_START:
        swa_model.update_parameters(model)
        swa_scheduler.step()
        if epoch == cfg.SWA_START:
            print(f"  SWA accumulation started")

    # ── Validation ───────────────────────────────────────────────────
    val_metrics = validate(model, val_loader, criterion, DEVICE)
    val_loss      = val_metrics['loss']
    val_acc       = val_metrics['acc']
    val_auc       = val_metrics['auc']
    val_eer       = val_metrics['eer']
    val_f1        = val_metrics['f1']
    val_precision = val_metrics['precision']
    val_recall    = val_metrics['recall']

    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['train_auc'].append(train_auc)
    history['train_eer'].append(train_eer)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    history['val_auc'].append(val_auc)
    history['val_eer'].append(val_eer)
    history['val_f1'].append(val_f1)
    history['val_precision'].append(val_precision)
    history['val_recall'].append(val_recall)
    history['lr'].append(current_lr)

    print(f"  Train | Loss:{train_loss:.4f} Acc:{train_acc:.4f}"
          f" AUC:{train_auc:.4f} EER:{train_eer:.4f}")
    print(f"  Val   | Loss:{val_loss:.4f} Acc:{val_acc:.4f}"
          f" AUC:{val_auc:.4f} EER:{val_eer:.4f}")
    print(f"        | F1:{val_f1:.4f} P:{val_precision:.4f}"
          f" R:{val_recall:.4f} LR:{current_lr:.2e}")

    if val_auc > best_val_auc:
        best_val_auc = val_auc
        best_val_eer = val_eer
        best_epoch   = epoch + 1
        patience_counter = 0
        mname = (f"best_cnn_model_fold{cfg.CURRENT_FOLD}.pth"
                 if cfg.CURRENT_FOLD >= 0 else "best_cnn_model.pth")
        torch.save(model.state_dict(),
                   os.path.join(cfg.OUTPUT_DIR, mname))
        print(f"  New best! AUC:{val_auc:.4f}")
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"\n  Early stopping at epoch {epoch+1}")
            break

# ── SWA: update BatchNorm and save ───────────────────────────────────
# Issue 10 FIX: Use final epoch instead of best_epoch for SWA check
if cfg.USE_SWA and epoch >= cfg.SWA_START:
    print("\n  Updating SWA BatchNorm statistics...")
    
    # CRITICAL BUG 2 FIX: PyTorch's update_bn expects (input, target) tuples
    # Our dataset returns dicts. Create a lightweight wrapper loader.
    class _BNUpdateWrapper:
        def __init__(self, loader):
            self.loader = loader
        def __iter__(self):
            for batch in self.loader:
                # FIX: Yield both frames and mask so BN stats update sees the true mask
                frames = batch['frames'].to(DEVICE)
                mask   = batch.get('mask', torch.ones(frames.shape[0], frames.shape[1], dtype=torch.bool)).to(DEVICE)
                yield frames, mask
        def __len__(self):
            return len(self.loader)
    
    # Issue 11 FIX: Use original_train_loader (unweighted) not current_train_loader
    wrapped_loader = _BNUpdateWrapper(original_train_loader)
    update_bn(wrapped_loader, swa_model, device=DEVICE)
    torch.save(swa_model.module.state_dict(),
               os.path.join(cfg.OUTPUT_DIR, "swa_model.pth"))
    print("  SWA model saved: swa_model.pth")

    # Check if SWA model is better
    val_metrics_swa = validate(
        swa_model.module, val_loader, criterion, DEVICE)
    if val_metrics_swa['auc'] > best_val_auc:
        best_val_auc = val_metrics_swa['auc']
        print(f"  SWA improved AUC to: {best_val_auc:.4f}")
        torch.save(swa_model.module.state_dict(),
                   os.path.join(cfg.OUTPUT_DIR,
                   f"best_cnn_model_fold{cfg.CURRENT_FOLD}.pth"))

print("\n" + "="*70)
print("TRAINING COMPLETE")
print(f"  Best Epoch : {best_epoch}")
print(f"  Best AUC   : {best_val_auc:.4f}")
print(f"  Best EER   : {best_val_eer:.4f}")
print("="*70)

# Issue 34 FIX: Add fold index to CSV filename
history_path = os.path.join(
    cfg.OUTPUT_DIR,
    f"training_history_fold{cfg.CURRENT_FOLD}.json"
    if cfg.CURRENT_FOLD >= 0 else "training_history.json")
with open(history_path, 'w') as f:
    json.dump(history, f, indent=2)
csv_path = os.path.join(
    cfg.OUTPUT_DIR,
    f"cnn_training_history_fold{cfg.CURRENT_FOLD}.csv"
    if cfg.CURRENT_FOLD >= 0 else "cnn_training_history.csv")
pd.DataFrame(history).to_csv(csv_path, index=False)
print(f"History saved: {history_path}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# TRAINING CURVES (Research-Grade Visualization)
# ═══════════════════════════════════════════════════════════════════════════════

import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

epochs = range(1, len(history['train_loss']) + 1)

# 1. Loss Curve
axes[0, 0].plot(epochs, history['train_loss'], 'b-', marker='o', label='Train', linewidth=2, markersize=4)
axes[0, 0].plot(epochs, history['val_loss'], 'r-', marker='s', label='Val', linewidth=2, markersize=4)
axes[0, 0].axvline(x=best_epoch, color='green', linestyle='--', alpha=0.7, label=f'Best (Epoch {best_epoch})')
axes[0, 0].set_xlabel('Epoch', fontsize=11)
axes[0, 0].set_ylabel('Loss', fontsize=11)
axes[0, 0].set_title('Loss Curve', fontsize=12, fontweight='bold')
axes[0, 0].legend(loc='upper right')
axes[0, 0].grid(True, alpha=0.3)

# 2. AUC Curve
axes[0, 1].plot(epochs, history['train_auc'], 'b-', marker='o', label='Train', linewidth=2, markersize=4)
axes[0, 1].plot(epochs, history['val_auc'], 'r-', marker='s', label='Val', linewidth=2, markersize=4)
axes[0, 1].axvline(x=best_epoch, color='green', linestyle='--', alpha=0.7, label=f'Best (Epoch {best_epoch})')
axes[0, 1].axhline(y=best_val_auc, color='green', linestyle=':', alpha=0.5)
axes[0, 1].set_xlabel('Epoch', fontsize=11)
axes[0, 1].set_ylabel('AUC-ROC', fontsize=11)
axes[0, 1].set_title(f'AUC Curve (Best: {best_val_auc:.4f})', fontsize=12, fontweight='bold')
axes[0, 1].legend(loc='lower right')
axes[0, 1].grid(True, alpha=0.3)
axes[0, 1].set_ylim([0.4, 1.0])

# 3. F1 / Precision / Recall
axes[1, 0].plot(epochs, history['val_f1'], 'g-', marker='o', label='F1', linewidth=2, markersize=4)
axes[1, 0].plot(epochs, history['val_precision'], 'b-', marker='^', label='Precision', linewidth=2, markersize=4)
axes[1, 0].plot(epochs, history['val_recall'], 'r-', marker='v', label='Recall', linewidth=2, markersize=4)
axes[1, 0].axvline(x=best_epoch, color='green', linestyle='--', alpha=0.7, label=f'Best Epoch')
axes[1, 0].set_xlabel('Epoch', fontsize=11)
axes[1, 0].set_ylabel('Score', fontsize=11)
axes[1, 0].set_title('Validation Metrics (F1, Precision, Recall)', fontsize=12, fontweight='bold')
axes[1, 0].legend(loc='lower right')
axes[1, 0].grid(True, alpha=0.3)
axes[1, 0].set_ylim([0.0, 1.0])

# 4. Learning Rate Schedule
axes[1, 1].plot(epochs, history['lr'], 'purple', marker='o', linewidth=2, markersize=4)
axes[1, 1].axvline(x=best_epoch, color='green', linestyle='--', alpha=0.7, label=f'Best Epoch')
axes[1, 1].set_xlabel('Epoch', fontsize=11)
axes[1, 1].set_ylabel('Learning Rate', fontsize=11)
axes[1, 1].set_title('Learning Rate Schedule (Linear Warmup + Cosine Decay)', fontsize=12, fontweight='bold')
axes[1, 1].set_yscale('log')
axes[1, 1].grid(True, alpha=0.3)
axes[1, 1].legend(loc='upper right')

# ISSUE 5 FIX: Warmup annotation accuracy
# Match the actual step-based warmup from training loop
# warmup_steps / steps_per_epoch gives the actual warmup duration in epochs
warmup_end_epoch = cfg.WARMUP_RATIO * (cfg.SWA_START if cfg.USE_SWA else cfg.NUM_EPOCHS)
if warmup_end_epoch > 0 and warmup_end_epoch <= len(epochs):
    axes[1, 1].axvspan(1, warmup_end_epoch, alpha=0.2, color='yellow', label='Warmup Phase')
    axes[1, 1].legend(loc='upper right')

plt.suptitle(f'CNN Spatio-Temporal Training Progress\n{cfg.EXPERIMENT_NAME} {cfg.EXPERIMENT_VERSION}', 
             fontsize=14, fontweight='bold')
plt.tight_layout()

# Save figure
fig_path = os.path.join(cfg.OUTPUT_DIR, f'training_curves_fold{cfg.CURRENT_FOLD}.png' if cfg.CURRENT_FOLD >= 0 else 'training_curves.png')
plt.savefig(fig_path, dpi=200, bbox_inches='tight', facecolor='white')
print(f"✓ Training curves saved to: {fig_path}")

plt.show()


## 6. Video-Level Inference & Export

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# LOAD BEST MODEL
# ═══════════════════════════════════════════════════════════════════════

possible_names = [
    f"best_cnn_model_fold{cfg.CURRENT_FOLD}.pth" if cfg.CURRENT_FOLD >= 0 else "",
    "best_cnn_model.pth",
]

model_path = None
for name in possible_names:
    if not name:
        continue
    temp = os.path.join(cfg.OUTPUT_DIR, name)
    if os.path.exists(temp):
        model_path = temp
        break

if model_path is None:
    pth_files = [f for f in os.listdir(cfg.OUTPUT_DIR) if f.endswith('.pth')]
    if pth_files:
        model_path = os.path.join(cfg.OUTPUT_DIR, pth_files[0])
    else:
        raise FileNotFoundError("No .pth model found. Run training cell first.")

model.load_state_dict(torch.load(model_path, map_location=DEVICE, weights_only=True))
model.eval()
print(f"✓ Best model loaded from: {model_path}")

In [ ]:
# Issue 18 FIX: predict_video_temporal and generate_video_predictions
# were never called - the TTA cell (cell 31) defines its own predict_video_tta.
# Removed dead code to avoid confusion.

print("Prediction functions defined in TTA cell (Cell 31)")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# VIDEO-LEVEL PREDICTIONS WITH TTA + PER-SOURCE BREAKDOWN
# ═══════════════════════════════════════════════════════════════════════════════

print("\n" + "="*70)
print("GENERATING FINAL VIDEO-LEVEL PREDICTIONS")
print("="*70)

# Load best model
best_model_path = os.path.join(
    cfg.OUTPUT_DIR,
    f"best_cnn_model_fold{cfg.CURRENT_FOLD}.pth"
    if cfg.CURRENT_FOLD >= 0 else "best_cnn_model.pth")
if os.path.exists(best_model_path):
    model.load_state_dict(torch.load(
        best_model_path, map_location=DEVICE, weights_only=True))
    print(f"✓ Best model loaded: {best_model_path}")
else:
    print(f"⚠️ No saved model found — using current state")


# Issue 29 FIX: Create transforms once outside the loop
_val_transforms_cached = get_val_transforms()

def predict_video_tta(model, cache_path, device, max_frames=15):
    """
    5-pass Test-Time Augmentation:
    1. Standard
    2. Horizontal flip
    3. Brightness +15
    4. Brightness -15
    5. Slight JPEG compression simulation
    Average gives more robust probability estimate.
    """
    if not os.path.exists(cache_path): return 0.5
    faces = np.load(cache_path)
    if len(faces) < 3: return 0.5

    n       = len(faces)
    step    = n / max_frames if n >= max_frames else 1
    indices = ([int(i * step) for i in range(max_frames)]
               if n >= max_frames else list(range(n)))
    while len(indices) < max_frames: indices.append(n-1)
    selected = [faces[i].astype('uint8') for i in indices]

    def run_pass(frames_list):
        tfs    = [_val_transforms_cached(image=f)['image']
                  for f in frames_list]
        fstack = torch.stack(tfs).unsqueeze(0).to(device)
        msk    = torch.ones(1, max_frames,
                            dtype=torch.bool).to(device)
        with torch.no_grad():
            return torch.sigmoid(model(fstack, msk)).item()

    probs = []

    # Pass 1: standard
    probs.append(run_pass(selected))

    # Pass 2: horizontal flip
    probs.append(run_pass(
        [np.fliplr(f).copy() for f in selected]))

    # Pass 3: brightness up
    probs.append(run_pass([
        np.clip(f.astype(np.int32)+15,0,255).astype(np.uint8)
        for f in selected]))

    # Pass 4: brightness down
    probs.append(run_pass([
        np.clip(f.astype(np.int32)-15,0,255).astype(np.uint8)
        for f in selected]))

    # Pass 5: slight blur (compression simulation)
    probs.append(run_pass([
        cv2.GaussianBlur(f,(3,3),0) for f in selected]))

    del faces

    # IMPROVEMENT 11: Simple uniform average — more robust than confidence-weighted
    # Confidence weighting amplifies miscalibration errors from augmented passes
    probs_arr = np.array(probs)
    return float(np.mean(probs_arr))


model.eval()
raw_preds = []
for video in tqdm(val_videos, desc="Predicting (TTA×5)"):
    vid_id = video['video_id']
    prob   = (predict_video_tta(model, cache_index[vid_id],
                                DEVICE, cfg.FRAMES_PER_VIDEO)
              if vid_id in cache_index else 0.5)
    raw_preds.append({
        'video_id'       : vid_id,
        'label'          : video['label'],
        'prediction'     : prob,
        'predicted_class': 1 if prob > 0.5 else 0,
        'source'         : video.get('source', 'unknown')
    })
    
gc.collect()
torch.cuda.empty_cache()

predictions_df = pd.DataFrame(raw_preds)
predictions_df.rename(columns={'prediction': 'P_CNN'}, inplace=True)

# ── PER-MANIPULATION-TYPE AUC BREAKDOWN (Research Requirement) ──────────────
print("\n" + "="*70)
print("PER-MANIPULATION-TYPE AUC BREAKDOWN")
print("="*70)
manipulation_types = ['Deepfakes', 'Face2Face', 'FaceSwap', 'NeuralTextures', 'FaceShifter', 'DeepFakeDetection']
for manip in manipulation_types:
    mask = predictions_df['source'].str.contains(manip, case=False, na=False)
    if mask.sum() >= 10:  # Need enough samples
        y_true_manip = predictions_df.loc[mask, 'label'].values
        y_pred_manip = predictions_df.loc[mask, 'P_CNN'].values
        if len(np.unique(y_true_manip)) > 1:
            manip_auc = roc_auc_score(y_true_manip, y_pred_manip)
            print(f"  {manip:20s}: AUC = {manip_auc:.4f} (n={mask.sum()})")
        else:
            print(f"  {manip:20s}: Single class only (n={mask.sum()})")
    else:
        print(f"  {manip:20s}: Insufficient samples (n={mask.sum()})")
print("="*70)

y_true       = predictions_df['label'].values
y_pred_proba = predictions_df['P_CNN'].values
y_pred       = (y_pred_proba > 0.5).astype(int)

video_acc       = accuracy_score(y_true, y_pred)
video_f1        = f1_score(y_true, y_pred, zero_division=0)
video_precision = precision_score(y_true, y_pred, zero_division=0)
video_recall    = recall_score(y_true, y_pred, zero_division=0)

if len(np.unique(y_true)) > 1:
    video_auc = roc_auc_score(y_true, y_pred_proba)
    video_eer = compute_eer(y_true, y_pred_proba)
    video_ap  = average_precision_score(y_true, y_pred_proba)
else:
    video_auc = video_eer = video_ap = 0.5

optimal_thresh, optimal_f1 = find_optimal_threshold(
    y_true, y_pred_proba)
y_pred_opt        = (y_pred_proba >= optimal_thresh).astype(int)
video_acc_optimal = accuracy_score(y_true, y_pred_opt)
video_f1_optimal  = f1_score(y_true, y_pred_opt, zero_division=0)

predictions_df.to_csv(
    os.path.join(cfg.OUTPUT_DIR, "cnn_predictions.csv"), index=False)

print(f"\n{'─'*50}")
print("OVERALL METRICS (threshold=0.5):")
print(f"  AUC-ROC   : {video_auc:.4f}")
print(f"  AP (PR)   : {video_ap:.4f}")
print(f"  Accuracy  : {video_acc:.4f}")
print(f"  F1-Score  : {video_f1:.4f}")
print(f"  Precision : {video_precision:.4f}")
print(f"  Recall    : {video_recall:.4f}")
print(f"  EER       : {video_eer:.4f}")
print(f"\nOPTIMAL THRESHOLD = {optimal_thresh:.2f}:")
print(f"  Accuracy  : {video_acc_optimal:.4f}")
print(f"  F1-Score  : {video_f1_optimal:.4f}")

# ── Per-source breakdown (critical for paper) ─────────────────────────
print(f"\n{'─'*50}")
print("PER-SOURCE BREAKDOWN:")
print(f"{'Source':<30} {'N':>5} {'AUC':>7} {'Acc':>7} {'F1':>7}")
print("─"*56)
for source in sorted(predictions_df['source'].unique()):
    s    = predictions_df[predictions_df['source']==source]
    st   = s['label'].values
    sp   = s['P_CNN'].values
    sy   = (sp > 0.5).astype(int)
    if len(np.unique(st)) > 1:
        sauc = roc_auc_score(st, sp)
        sacc = accuracy_score(st, sy)
        sf1  = f1_score(st, sy, zero_division=0)
        print(f"  {source:<28} {len(s):>5} {sauc:>7.4f}"
              f" {sacc:>7.4f} {sf1:>7.4f}")
    else:
        print(f"  {source:<28} {len(s):>5} {'single class':>22}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# STEP 2: BOOTSTRAP CONFIDENCE INTERVALS (Research Requirement)
# ═══════════════════════════════════════════════════════════════════════════════

def bootstrap_ci(y_true, y_pred, metric_fn, n_bootstrap=1000, ci=95, seed=42):
    """Compute bootstrap confidence intervals for a metric."""
    rng = np.random.RandomState(seed)
    n = len(y_true)
    scores = []
    for _ in range(n_bootstrap):
        indices = rng.randint(0, n, n)
        y_true_boot = y_true[indices]
        y_pred_boot = y_pred[indices]
        if len(np.unique(y_true_boot)) < 2:
            continue
        try:
            scores.append(metric_fn(y_true_boot, y_pred_boot))
        except Exception:
            continue
    if len(scores) == 0:
        point = metric_fn(y_true, y_pred)
        return point, point, point
    scores = np.array(scores)
    alpha = (100 - ci) / 2
    return np.percentile(scores, alpha), metric_fn(y_true, y_pred), np.percentile(scores, 100 - alpha)

print("\n" + "="*70)
print("BOOTSTRAP CONFIDENCE INTERVALS (1000 iterations)")
print("="*70)

auc_low, auc_mid, auc_high = bootstrap_ci(y_true, y_pred_proba, roc_auc_score)
acc_low, acc_mid, acc_high = bootstrap_ci(y_true, y_pred, accuracy_score)
f1_low, f1_mid, f1_high   = bootstrap_ci(y_true, y_pred, f1_score)
# FIXED: EER now also has bootstrap CI
eer_low, eer_mid, eer_high = bootstrap_ci(y_true, y_pred_proba, compute_eer)

print(f"\n  AUC-ROC:   {auc_mid:.4f} [95% CI: {auc_low:.4f}, {auc_high:.4f}]")
print(f"  Accuracy:  {acc_mid:.4f} [95% CI: {acc_low:.4f}, {acc_high:.4f}]")
print(f"  F1-Score:  {f1_mid:.4f} [95% CI: {f1_low:.4f}, {f1_high:.4f}]")
print(f"  EER:       {eer_mid:.4f} [95% CI: {eer_low:.4f}, {eer_high:.4f}]")

# Save metrics with CI — all 4 metrics now have intervals
metrics_df = pd.DataFrame([
    {'metric': 'AUC-ROC',  'value': auc_mid,  'ci_low': auc_low,  'ci_high': auc_high},
    {'metric': 'Accuracy', 'value': acc_mid,  'ci_low': acc_low,  'ci_high': acc_high},
    {'metric': 'F1-Score', 'value': f1_mid,   'ci_low': f1_low,   'ci_high': f1_high},
    {'metric': 'EER',      'value': eer_mid,  'ci_low': eer_low,  'ci_high': eer_high},
])
metrics_df.to_csv(os.path.join(cfg.OUTPUT_DIR, "cnn_metrics_with_ci.csv"), index=False)
print(f"\n✓ Metrics with 95% CI saved to: cnn_metrics_with_ci.csv")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# STEP 3: RESEARCH-GRADE VISUALIZATIONS
# ═══════════════════════════════════════════════════════════════════════════════

import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, precision_recall_curve, average_precision_score, roc_curve
import seaborn as sns

# Prepare data
real_preds = predictions_df[predictions_df['label'] == 0]['P_CNN']
fake_preds = predictions_df[predictions_df['label'] == 1]['P_CNN']

fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# 1. ROC Curve
fpr, tpr, _ = roc_curve(y_true, y_pred_proba)
axes[0, 0].plot(fpr, tpr, 'b-', linewidth=2.5, label=f'CNN (AUC = {video_auc:.4f})')
axes[0, 0].fill_between(fpr, tpr, alpha=0.2, color='blue')
axes[0, 0].plot([0, 1], [0, 1], 'k--', alpha=0.5, linewidth=1.5, label='Random')
axes[0, 0].set_xlabel('False Positive Rate', fontsize=12)
axes[0, 0].set_ylabel('True Positive Rate', fontsize=12)
axes[0, 0].set_title('ROC Curve', fontsize=14, fontweight='bold')
axes[0, 0].legend(loc='lower right')
axes[0, 0].grid(True, alpha=0.3)

# 2. Precision-Recall Curve
precision_arr, recall_arr, _ = precision_recall_curve(y_true, y_pred_proba)
ap = average_precision_score(y_true, y_pred_proba)
axes[0, 1].plot(recall_arr, precision_arr, 'g-', linewidth=2.5, label=f'CNN (AP = {ap:.4f})')
axes[0, 1].fill_between(recall_arr, precision_arr, alpha=0.2, color='green')
axes[0, 1].axhline(y=y_true.mean(), color='k', linestyle='--', alpha=0.5, label=f'No Skill ({y_true.mean():.2f})')
axes[0, 1].set_xlabel('Recall', fontsize=12)
axes[0, 1].set_ylabel('Precision', fontsize=12)
axes[0, 1].set_title('Precision-Recall Curve', fontsize=14, fontweight='bold')
axes[0, 1].legend(loc='lower left')
axes[0, 1].grid(True, alpha=0.3)

# 3. Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[1, 0],
            xticklabels=['Real', 'Fake'], yticklabels=['Real', 'Fake'],
            annot_kws={'size': 16})
axes[1, 0].set_xlabel('Predicted', fontsize=12)
axes[1, 0].set_ylabel('Actual', fontsize=12)
axes[1, 0].set_title('Confusion Matrix', fontsize=14, fontweight='bold')

# 4. Score Distribution
axes[1, 1].hist(real_preds, bins=30, alpha=0.7, label='Real', color='green', density=True)
axes[1, 1].hist(fake_preds, bins=30, alpha=0.7, label='Fake', color='red', density=True)
axes[1, 1].axvline(x=0.5, color='black', linestyle='--', linewidth=2, label='Threshold (0.5)')
axes[1, 1].axvline(x=optimal_thresh, color='orange', linestyle='--', linewidth=2, label=f'Optimal ({optimal_thresh:.2f})')
axes[1, 1].set_xlabel('Predicted Probability (Fake)', fontsize=12)
axes[1, 1].set_ylabel('Density', fontsize=12)
axes[1, 1].set_title('Score Distribution by Class', fontsize=14, fontweight='bold')
axes[1, 1].legend(loc='upper center')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(cfg.OUTPUT_DIR, 'cnn_evaluation_plots.png'), dpi=150, bbox_inches='tight')
plt.show()

print("\n✓ Evaluation plots saved to: cnn_evaluation_plots.png")

# Classification Report
print("\n" + "="*70)
print("CLASSIFICATION REPORT")
print("="*70)
print(classification_report(y_true, y_pred, target_names=['Real', 'Fake'], digits=4))


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# GRAD-CAM VISUALIZATION (Fixed for Xception)
# ═══════════════════════════════════════════════════════════════════════════════

import torch.nn.functional as F

class SimpleGradCAM:
    """Grad-CAM for temporal models — P100 safe, Xception compatible."""

    def __init__(self, cnn_model):
        self.model       = cnn_model
        self.gradients   = None
        self.activations = None
        self.handles     = []

        # ISSUE 6 FIX: Xception uses 'act4' (final activation), EfficientNet uses 'conv_head'
        try:
            target_layer = self.model.backbone.act4     # Issue 8 FIX: Xception final activation
            print("✓ Grad-CAM: using Xception act4 layer (final activation)")
        except AttributeError:
            try:
                target_layer = self.model.backbone.conv_head  # EfficientNet
                print("✓ Grad-CAM: using EfficientNet conv_head layer")
            except AttributeError:
                # Generic fallback — last conv layer
                convs = [(n,m) for n,m in self.model.backbone.named_modules()
                         if isinstance(m, nn.Conv2d)]
                target_layer = convs[-1][1]
                print(f"✓ Grad-CAM: fallback to last conv layer")

        self.handles.append(
            target_layer.register_forward_hook(self._save_activation))
        self.handles.append(
            target_layer.register_full_backward_hook(self._save_gradient))

    def _save_activation(self, module, input, output):
        self.activations = output.detach()

    def _save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0].detach()

    def cleanup(self):
        for h in self.handles:
            h.remove()
        self.handles = []

    def generate(self, frames, mask, target_frame_idx=0):
        self.model.eval()

        logit = self.model(frames, mask)
        prob = torch.sigmoid(logit).detach().item()

        self.model.zero_grad()
        logit.squeeze().backward(retain_graph=True)

        if self.gradients is None or self.activations is None:
            return np.zeros(frames.shape[3:]), prob

        target_grads = self.gradients[target_frame_idx:target_frame_idx + 1]
        target_acts  = self.activations[target_frame_idx:target_frame_idx + 1]

        weights = target_grads.mean(dim=(2, 3), keepdim=True)
        cam     = F.relu((weights * target_acts).sum(dim=1, keepdim=True))

        if cam.max() - cam.min() > 1e-8:
            cam = (cam - cam.min()) / (cam.max() - cam.min())
        else:
            cam = torch.zeros_like(cam)

        cam = F.interpolate(cam, size=frames.shape[3:],
                            mode='bilinear', align_corners=False)

        # P100 FIX: use .tolist() NOT .numpy() — avoids C-API crash
        cam_numpy = np.array(cam.squeeze().cpu().tolist())
        return cam_numpy, prob


print("\n" + "="*70)
print("GENERATING GRAD-CAM VISUALIZATIONS")
print("="*70)

gradcam = SimpleGradCAM(model)

# Collect up to 3 valid videos first, THEN draw the figure
valid_videos_for_gradcam = []
for video in val_videos:
    if len(valid_videos_for_gradcam) >= 3:
        break
    vid_id = video['video_id']
    if vid_id not in cache_index:
        continue
    faces_check = np.load(cache_index[vid_id])
    if len(faces_check) < 5:
        del faces_check
        continue
    valid_videos_for_gradcam.append((video, faces_check))

if len(valid_videos_for_gradcam) == 0:
    print("⚠️ No valid videos found for Grad-CAM visualization")
else:
    # Create figure
    fig, axes = plt.subplots(len(valid_videos_for_gradcam), 5,
                             figsize=(18, 4*len(valid_videos_for_gradcam)))
    if len(valid_videos_for_gradcam) == 1:
        axes = [axes]

    # ISSUE 10 FIX: Generate indices dynamically based on actual FRAMES_PER_VIDEO
    frame_indices = np.linspace(0, cfg.FRAMES_PER_VIDEO - 1, 5, dtype=int).tolist()
    
    val_transforms = get_val_transforms()

    for row_idx, (video, faces) in enumerate(valid_videos_for_gradcam):
        vid_id = video['video_id']
        label = video['label']

        # Sample frames uniformly
        n_faces = len(faces)
        if n_faces >= cfg.FRAMES_PER_VIDEO:
            step = n_faces / cfg.FRAMES_PER_VIDEO
            indices_to_use = [int(i * step) for i in range(cfg.FRAMES_PER_VIDEO)]
        else:
            indices_to_use = list(range(n_faces))
            while len(indices_to_use) < cfg.FRAMES_PER_VIDEO:
                indices_to_use.append(n_faces - 1)

        selected_faces = [faces[i] for i in indices_to_use]

        # Apply transforms
        frame_tensors = []
        for face in selected_faces:
            if hasattr(face, 'astype'):
                face = face.astype('uint8')
            aug = val_transforms(image=face)
            frame_tensors.append(aug['image'])

        frames_tensor = torch.stack(frame_tensors).unsqueeze(0).to(DEVICE)
        mask_tensor = torch.ones(1, cfg.FRAMES_PER_VIDEO, dtype=torch.bool).to(DEVICE)

        for col_idx, frame_idx in enumerate(frame_indices):
            if frame_idx >= len(selected_faces):
                frame_idx = len(selected_faces) - 1

            cam, prob = gradcam.generate(frames_tensor, mask_tensor, frame_idx)

            # Get original face for display
            face_display = selected_faces[frame_idx]
            if hasattr(face_display, 'astype'):
                face_display = face_display.astype('uint8')

            ax = axes[row_idx][col_idx]
            ax.imshow(face_display)

            # Overlay CAM
            ax.imshow(cam, cmap='jet', alpha=0.4)

            # Title
            label_str = "FAKE" if label == 1 else "REAL"
            ax.set_title(f"{label_str} | Frame {frame_idx} | P={prob:.2f}", fontsize=10)
            ax.axis('off')

        del faces  # Free memory

    plt.tight_layout()
    plt.savefig(os.path.join(cfg.OUTPUT_DIR, 'gradcam_visualization.png'),
                dpi=150, bbox_inches='tight')
    plt.show()
    print(f"✓ Grad-CAM saved: gradcam_visualization.png")

# Cleanup
gradcam.cleanup()
gc.collect()
torch.cuda.empty_cache()

## 7. Late Fusion Integration Guide

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# LATE FUSION INTEGRATION GUIDE
# ═══════════════════════════════════════════════════════════════════════════════

print("""
╔══════════════════════════════════════════════════════════════════════════════╗
║                       LATE FUSION INTEGRATION GUIDE                          ║
╠══════════════════════════════════════════════════════════════════════════════╣
║                                                                              ║
║  This notebook outputs: cnn_predictions.csv                                  ║
║  Columns: video_id, P_CNN                                                    ║
║                                                                              ║
║  To fuse with rPPG predictions from 2ND_MODEL.ipynb:                        ║
║                                                                              ║
║  ┌─────────────────────────────────────────────────────────────────────────┐ ║
║  │  # Load both predictions                                                │ ║
║  │  cnn_df = pd.read_csv('cnn_predictions.csv')                           │ ║
║  │  rppg_df = pd.read_csv('rppg_predictions.csv')                         │ ║
║  │                                                                         │ ║
║  │  # Merge on video_id                                                    │ ║
║  │  fused_df = cnn_df.merge(rppg_df, on='video_id')                       │ ║
║  │                                                                         │ ║
║  │  # Simple average fusion                                                │ ║
║  │  fused_df['P_final'] = (fused_df['P_CNN'] + fused_df['P_rPPG']) / 2    │ ║
║  │                                                                         │ ║
║  │  # Weighted fusion (if CNN is more accurate)                           │ ║
║  │  w_cnn, w_rppg = 0.6, 0.4                                              │ ║
║  │  fused_df['P_final'] = w_cnn * fused_df['P_CNN']                       │ ║
║  │                      + w_rppg * fused_df['P_rPPG']                      │ ║
║  │                                                                         │ ║
║  │  # Learned fusion (train a small classifier)                           │ ║
║  │  from sklearn.linear_model import LogisticRegression                   │ ║
║  │  X_fusion = fused_df[['P_CNN', 'P_rPPG']].values                       │ ║
║  │  y_fusion = fused_df['label'].values                                   │ ║
║  │  fusion_model = LogisticRegression().fit(X_fusion, y_fusion)           │ ║
║  │  fused_df['P_final'] = fusion_model.predict_proba(X_fusion)[:, 1]      │ ║
║  └─────────────────────────────────────────────────────────────────────────┘ ║
║                                                                              ║
╚══════════════════════════════════════════════════════════════════════════════╝
""")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# FINAL SUMMARY
# ═══════════════════════════════════════════════════════════════════════════════

print("\n" + "="*70)
print("CNN SPATIO-TEMPORAL STREAM — FINAL SUMMARY")
print("="*70)

# Get parameter counts (handle both variable names)
total = total_params if 'total_params' in vars() else sum(p.numel() for p in model.parameters())
trainable_count = trainable if 'trainable' in vars() else sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"""
┌─────────────────────────────────────────────────────────────────────────────┐
│  ARCHITECTURE                                                               │
├─────────────────────────────────────────────────────────────────────────────┤
│  Backbone:           Xception (ImageNet pretrained)                  │
│  Temporal Model:     BiLSTM + Multi-Head Self-Attention                     │
│  LSTM Hidden:        {cfg.LSTM_HIDDEN} × 2 (bidirectional)                                │
│  LSTM Layers:        {cfg.LSTM_LAYERS}                                                     │
│  Attention Heads:    {cfg.ATTENTION_HEADS}                                                     │
│  Total Parameters:   {total:,}                                       │
│  Trainable Params:   {trainable_count:,}                                       │
└─────────────────────────────────────────────────────────────────────────────┘

┌─────────────────────────────────────────────────────────────────────────────┐
│  TRAINING CONFIGURATION                                                     │
├─────────────────────────────────────────────────────────────────────────────┤
│  Training Videos:    {len(train_videos)}                                                       │
│  Validation Videos:  {len(val_videos)}                                                       │
│  Frames per Video:   {cfg.FRAMES_PER_VIDEO}                                                        │
│  Batch Size:         {cfg.BATCH_SIZE} (effective: {cfg.BATCH_SIZE * cfg.GRAD_ACCUMULATION_STEPS})                                         │
│  Learning Rate:      {cfg.LEARNING_RATE}                                                    │
│  Loss Function:      Focal Loss (α={cfg.FOCAL_ALPHA}, γ={cfg.FOCAL_GAMMA})                              │
└─────────────────────────────────────────────────────────────────────────────┘

┌─────────────────────────────────────────────────────────────────────────────┐
│  BEST FRAME-LEVEL RESULTS                                                   │
├─────────────────────────────────────────────────────────────────────────────┤
│  Best Epoch:         {best_epoch}                                                        │
│  Best Val AUC:       {best_val_auc:.4f}                                                    │
│  Best Val EER:       {best_val_eer:.4f}                                                    │
└─────────────────────────────────────────────────────────────────────────────┘

┌─────────────────────────────────────────────────────────────────────────────┐
│  VIDEO-LEVEL METRICS (Final Evaluation)                                     │
├─────────────────────────────────────────────────────────────────────────────┤
│  AUC-ROC:            {video_auc:.4f}                                                    │
│  Accuracy:           {video_acc:.4f}                                                    │
│  F1-Score:           {video_f1:.4f}                                                    │
│  EER:                {video_eer:.4f}                                                    │
└─────────────────────────────────────────────────────────────────────────────┘

┌─────────────────────────────────────────────────────────────────────────────┐
│  OUTPUT FILES                                                               │
├─────────────────────────────────────────────────────────────────────────────┤
│  ✓ cnn_predictions.csv          (video-level P_CNN scores for Late Fusion) │
│  ✓ best_cnn_model.pth           (best checkpoint weights)                  │
│  ✓ cnn_spatial_stream_final.pth (final model weights)                      │
│  ✓ cnn_training_history.csv     (epoch-by-epoch metrics)                   │
│  ✓ cnn_config.csv               (hyperparameters & final results)          │
└─────────────────────────────────────────────────────────────────────────────┘
""")


# Save final model weights (distinct from best checkpoint)
final_model_path = os.path.join(cfg.OUTPUT_DIR, "cnn_spatial_stream_final.pth")
torch.save(model.state_dict(), final_model_path)
print(f"✓ Final model saved: {final_model_path}")

# Save config as CSV for paper reproducibility
config_csv_path = os.path.join(cfg.OUTPUT_DIR, "cnn_config.csv")
pd.DataFrame([cfg.to_dict()]).to_csv(config_csv_path, index=False)
print(f"✓ Config saved as CSV: {config_csv_path}")


print("="*70)
print("✅ CNN SPATIO-TEMPORAL STREAM TRAINING COMPLETE!")
print("="*70)
print("\n  Key achievements:")
print("  ✓ Temporal modeling via BiLSTM + Multi-Head Attention")
print("  ✓ Detects inter-frame artifacts (flickering, blending shifts)")
print("  ✓ P100 compatible (FP32 only, no AMP)")
print("  ✓ Ready for Late Fusion with rPPG physiological stream")
print("\n  Next step:")
print("  → Run final_MODEL_rppg.ipynb to get P_rPPG scores")
print("  → Combine P_CNN + P_rPPG for Late Fusion ensemble")
print("="*70)

